# Advanced Metaheuristics for the 0/1 Knapsack Problem
## Comparative Study: Genetic Algorithm, Tabu Search, and Local Search

---

### ✨ Key Improvements in This Notebook

This enhanced version includes:

1. **🔧 Dedicated GA Configuration Class** (`GeneticAlgorithmConfig`)
   - Adaptive parameter scaling based on instance size
   - Proper encapsulation of GA parameters
   - Dynamic mutation rate adjustment
   - Population diversity tracking

2. **📖 Enhanced Documentation**
   - Detailed GA component explanations with visual examples
   - Comprehensive pseudocode and algorithms
   - Parameter justification tables
   - Expected performance characteristics

3. **🎨 Improved Organization**
   - Emoji-enhanced section titles for better navigation
   - Clearer part descriptions
   - Professional structure for educational use

---

### 📚 Learning Objectives

By the end of this lab session, you will be able to:

1. **Understand** the 0/1 Knapsack Problem and its complexity
2. **Implement** Local Search, Tabu Search, and Genetic Algorithm metaheuristics
3. **Compare** metaheuristic solutions with exact optimal solutions (Gurobi)
4. **Analyze** the trade-offs between solution quality and computational time
5. **Visualize** convergence behavior and understand different search strategies
6. **Understand** how Genetic Algorithm uses population-based evolution
7. **Understand** how Tabu Search uses memory to escape local optima

---

### 🎯 Problem Definition: The 0/1 Knapsack Problem

The **0/1 Knapsack Problem** is a classic **NP-hard** combinatorial optimization problem with applications in resource allocation, portfolio optimization, and decision-making.

**Given:**
- A set of $n$ items: $N = \{1, 2, \dots, n\}$
- Each item $i$ has:
  - A value: $v_i > 0$
  - A weight: $w_i > 0$
- A knapsack with capacity: $W > 0$

**Goal:** Select a subset of items to maximize total value without exceeding the weight capacity.

**Mathematical Formulation:**

$$\max Z = \sum_{i=1}^{n} v_i x_i$$

$$\text{subject to: } \sum_{i=1}^{n} w_i x_i \leq W$$

$$x_i \in \{0, 1\}, \quad \forall i \in N$$

where $x_i = 1$ if item $i$ is selected, and $x_i = 0$ otherwise.

---
### 🔬 Algorithms Implemented
### Algorithms Implemented
1. **Gurobi Solver** - Exact optimization (provides optimal baseline for comparison)
2. **Local Search (LS)** - Single-solution trajectory methods:
   - Best Improvement (Steepest Descent)
   - First Improvement (Fast exploration)

3. **Tabu Search (TS)** - Memory-based metaheuristic with adaptive search
4. **Genetic Algorithm (GA)** - Population-based evolutionary approach with selection, crossover, and mutation 
5. **Comparison Focus**: Solution quality, computational efficiency, and convergence behavior


---

## 📋 Part 1: Experimental Setup and Configuration


### 1.1 Required Libraries and Dependencies

In this section, we import all necessary libraries and modules required for the implementation of the heuristics and data handling. Additional libraries may be imported as needed for specific functionalities. We also define the output directories to organize the results, plots, and statistics generated during the experiments.


In [ ]:
import numpy as np # For numerical operations
import matplotlib.pyplot as plt # For plotting
import time # For timing operations
import os # For operating system interactions
from gurobipy import GRB, Model, quicksum # For optimization modeling
import openpyxl as pxl # For Excel file manipulation
from openpyxl.styles import Font, PatternFill, Alignment # For styling Excel cells
import pandas as pd # For data manipulation
from collections import defaultdict # For default dictionary
import gc  # For garbage collection to manage memory
import warnings  # For warning suppression
warnings.filterwarnings('ignore')

# Optional: Try to import tqdm for progress bars
try:
    from tqdm import tqdm
    TQDM_AVAILABLE = True
    print("✓ tqdm available - progress bars enabled")
except ImportError:
    TQDM_AVAILABLE = False
    print("ℹ tqdm not available - progress bars disabled (install with: pip install tqdm)")

# Create necessary directories
os.makedirs('plots', exist_ok=True) # For convergence plots
os.makedirs('statistics', exist_ok=True) # For detailed Excel statistics
os.makedirs('results', exist_ok=True) # For summary reports

print("✓ All libraries imported successfully")
print("✓ Output directories created: plots/, statistics/, results/")

### 1.2 Experimental Testing Configuration

**Purpose**: Control the scope and scale of experimental evaluation.

**Options**:
- `max_instances = 50`: Quick test (recommended for lab - takes ~1-2 minutes)
- `max_instances = 1000`: Full test (comprehensive - takes hours)

**Note**: You can change this value at any time and re-run the notebook.

In [ ]:
# Configuration: Number of instances to test
max_instances = 50  # Change to 1000 for full testing

print(f"Configuration: Testing up to {max_instances} instances")
print(f"Mode: {'FULL TEST (all instances)' if max_instances >= 1000 else 'DEFAULT TEST (sample instances)'}")

### 1.3 Algorithm Configuration Class

**Purpose**: Centralize and standardize algorithm parameters for fair comparison.

**Key Principles**:
1. **Time Budget Fairness**: All algorithms get equal maximum runtime
2. **Instance-Size Scaling**: Parameters scale with problem size
3. **Computational Budget**: Iteration limits based on neighborhood size
4. **Consistency**: Same neighborhood structures across algorithms

**Parameters**:
- `time_budget`: Maximum time per algorithm (default: 60 seconds)
- `max_evaluations`: Scales with instance size (n × multiplier)
- `tabu_tenure`: Length of tabu list (7-15 depending on instance size)
- `diversification_threshold`: Number of iterations without improvement before diversification

In [ ]:
class AlgorithmConfig:
    """
    Configuration class for algorithm parameters.
    Ensures fair comparison by scaling parameters with instance size.
    """
    
    def __init__(self, n_items, time_budget=60):
        """
        Initialize configuration based on instance size.
        
        Args:
            n_items: Number of items in the instance
            time_budget: Maximum time budget per algorithm (seconds)
        """
        self.n_items = n_items
        self.time_budget = time_budget
        
        # Scale iterations with problem size
        # For smaller instances: more thorough search
        # For larger instances: limited by computational budget
        if n_items <= 100:
            self.ls_max_iter = 10000
            self.ts_max_iter = 1000
        elif n_items <= 1000:
            self.ls_max_iter = 5000
            self.ts_max_iter = 500
        elif n_items <= 10000:
            self.ls_max_iter = 1000
            self.ts_max_iter = 200
        else:  # Very large instances
            self.ls_max_iter = 500
            self.ts_max_iter = 100
        
        # Local search iterations within Tabu Search (for initial solution)
        self.ls_inner_max_iter = min(500, self.ls_max_iter // 2)
        
        # Tabu Search parameters
        # Tabu tenure: typically 5-10 for small instances, higher for large
        if n_items <= 100:
            self.tabu_tenure = 7
        elif n_items <= 1000:
            self.tabu_tenure = 10
        else:
            self.tabu_tenure = 15
        
        # Diversification: number of iterations without improvement before restarting
        self.diversification_threshold = min(50, self.ts_max_iter // 10)
        
    def __str__(self):
        """String representation for debugging"""
        return f"""AlgorithmConfig(n_items={self.n_items}):
  - LS max_iter: {self.ls_max_iter}
  - TS max_iter: {self.ts_max_iter}
  - LS inner max_iter: {self.ls_inner_max_iter}
  - Tabu tenure: {self.tabu_tenure}
  - Diversification threshold: {self.diversification_threshold}
  - Time budget: {self.time_budget}s"""

print("✓ AlgorithmConfig class defined\n")

### 1.3.1 Genetic Algorithm Configuration Class

**Purpose**: Dedicated configuration for Genetic Algorithm parameters with adaptive scaling.

**Key Features**:
1. **Population Size Scaling**: Adapts to instance complexity
2. **Generation Budget**: Balanced exploration-exploitation trade-off
3. **Operator Probabilities**: Fine-tuned for knapsack problem
4. **Elitism Strategy**: Preserve best solutions across generations

**Design Principles**:
- Larger populations for small instances (more thorough search)
- Fewer generations for large instances (computational efficiency)

- Adaptive mutation rate (inversely proportional to problem size)
- Tournament size balances selection pressure

In [ ]:
class GeneticAlgorithmConfig:
    """
    Dedicated configuration class for Genetic Algorithm parameters.
    Provides adaptive parameter scaling based on instance characteristics.
    """
    
    def __init__(self, n_items, time_budget=60):
        """
        Initialize GA configuration based on problem instance size.
        
        Args:
            n_items: Number of items in the knapsack instance
            time_budget: Maximum time budget in seconds
        """
        self.n_items = n_items
        self.time_budget = time_budget
        
        # Population size: Decreases with instance size
        # Small instances: Larger population for thorough exploration
        # Large instances: Smaller population for computational efficiency
        if n_items <= 100:
            self.population_size = 100
            self.max_generations = 200
            self.tournament_size = 5
        elif n_items <= 500:
            self.population_size = 80
            self.max_generations = 150
            self.tournament_size = 4
        elif n_items <= 1000:
            self.population_size = 50
            self.max_generations = 100
            self.tournament_size = 3
        elif n_items <= 5000:
            self.population_size = 40
            self.max_generations = 75
            self.tournament_size = 3
        else:  # Very large instances
            self.population_size = 30
            self.max_generations = 50
            self.tournament_size = 3
        
        # Genetic operator probabilities
        self.crossover_rate = 0.85  # High crossover for building block exchange
        
        # Adaptive mutation rate: Decreases with problem size
        # Formula: max(min_rate, base_rate / n_items)
        self.mutation_rate = max(0.01, min(0.1, 2.0 / n_items))
        
        # Elitism: Keep top 5% of population (at least 1 individual)
        self.elitism_count = max(1, self.population_size // 20)
        
        # Diversity management
        self.diversity_threshold = 0.1  # Min diversity ratio to trigger restart
        
        # Crossover strategies available
        self.crossover_strategies = ['single_point', 'two_point', 'uniform']
        self.default_crossover = 'single_point'  # Best for knapsack
        
        # Mutation strategies
        self.mutation_strategies = ['bit_flip', 'swap', 'adaptive']
        self.default_mutation = 'bit_flip'  # Standard bit-flip mutation
        
        # Selection pressure
        # Higher tournament size = more selection pressure
        self.selection_pressure = 'medium'  # low, medium, high
        
    def get_adaptive_mutation_rate(self, generation, max_generations):
        """
        Calculate adaptive mutation rate that increases over generations.
        Helps escape local optima in later stages.
        
        Args:
            generation: Current generation number
            max_generations: Total number of generations
        
        Returns:
            Adaptive mutation rate
        """
        # Linear increase from base_rate to 2*base_rate
        progress = generation / max_generations
        return self.mutation_rate * (1.0 + progress)
    
    def get_diversity_score(self, population):
        """
        Calculate population diversity as ratio of unique individuals.
        
        Args:
            population: List of binary solution vectors
        
        Returns:
            Diversity score (0.0 to 1.0)
        """
        unique_individuals = len(set([tuple(ind) for ind in population]))
        return unique_individuals / len(population)
    
    def __str__(self):
        """String representation for debugging and logging"""
        return f"""GeneticAlgorithmConfig(n_items={self.n_items}):
  Population Parameters:
    - Population size: {self.population_size}
    - Max generations: {self.max_generations}
    - Elitism count: {self.elitism_count} ({self.elitism_count/self.population_size*100:.1f}%)
  
  Genetic Operators:
    - Crossover rate: {self.crossover_rate:.2f} ({self.crossover_rate*100:.0f}%)
    - Mutation rate: {self.mutation_rate:.4f} ({self.mutation_rate*100:.2f}%)
    - Tournament size: {self.tournament_size}
  
  Search Strategy:
    - Crossover type: {self.default_crossover}
    - Mutation type: {self.default_mutation}
    - Selection pressure: {self.selection_pressure}
    - Diversity threshold: {self.diversity_threshold:.2f}
  
  Resource Limits:
    - Time budget: {self.time_budget}s
    - Max evaluations: ~{self.population_size * self.max_generations:,}
"""

print("✓ GeneticAlgorithmConfig class defined")
print("\n📊 Example configuration for different instance sizes:")
for size in [100, 500, 1000, 5000]:
    config = GeneticAlgorithmConfig(size)
    print(f"\nInstance size {size}:")
    print(f"  Population: {config.population_size}, Generations: {config.max_generations}")
    print(f"  Mutation rate: {config.mutation_rate:.4f}, Tournament: {config.tournament_size}")

### 1.4 Instance Management Functions

**Purpose**: Define functions to handle knapsack problem instances efficiently.

**Key Features**:
- **Memory Efficient**: Load instances one at a time (not all at once)
- **Lazy Loading**: Instance data loaded only when needed
- **Scalable**: Can handle hundreds of instances without memory issues

**Functions**:
1. `get_instance_list()`: Lists available instance files without loading data
2. `load_single_instance()`: Loads a single instance from Excel file
3. `extract_instance_info()`: Extracts metadata (size) from filename

In [ ]:
def get_instance_list(directory='kp_instances', max_count=None):
    """
    Get list of instance files without loading them.
    
    Args:
        directory: Directory containing instance files
        max_count: Maximum number of instances to list (None = all)
    
    Returns:
        List of (filename, filepath) tuples
    """
    instances = []
    for filename in sorted(os.listdir(directory)):
        if filename.endswith('.xlsx'):
            filepath = os.path.join(directory, filename)
            instances.append((filename, filepath))
            if max_count and len(instances) >= max_count:
                break
    return instances

def load_single_instance(filepath):
    """
    Load a single instance from file.
    Memory efficient - loads only when needed.
    
    Args:
        filepath: Path to the instance file
    
    Returns:
        Dictionary with 'weights', 'values', 'capacity', 'n_items'
    """
    wb = pxl.load_workbook(filepath, read_only=True, data_only=True)
    sheet = wb.active
    
    weights = []
    values = []
    capacity = None
    
    for row in sheet.iter_rows(min_row=2, values_only=True):
        if row[0] is not None and row[1] is not None:
            weights.append(int(row[0]))
            values.append(int(row[1]))
        if row[2] is not None and capacity is None:
            capacity = int(row[2])
    
    wb.close()
    
    return {
        'weights': np.array(weights),
        'values': np.array(values),
        'capacity': capacity,
        'n_items': len(weights)
    }

def extract_instance_info(filename):
    """
    Extract size information from filename.
    Example: 'kp_100_items_1.xlsx' -> size=100
    """
    try:
        parts = filename.replace('.xlsx', '').split('_')
        for i, part in enumerate(parts):
            if 'items' in part.lower() or part.isdigit():
                if part.isdigit():
                    return int(part)
                elif i > 0 and parts[i-1].isdigit():
                    return int(parts[i-1])
        return 0
    except:
        return 0

print("✓ Instance management functions defined")

# # Test the instance loading functions
# instances = get_instance_list(max_count=10)
# for fname, fpath in instances:
#     data = load_single_instance(fpath)
#     size = extract_instance_info(fname)
#     print(f"Loaded instance '{fname}' with size {size}:")
#     print(f"  Number of items: {data['n_items']}")
#     print(f"  Capacity: {data['capacity']}")
#     print(f"  Weights: {data['weights'][:5]}...")  # Show first 5 weights
#     print(f"  Values: {data['values'][:5]}...")    # Show first 5 values
# print("✓ Instance loading functions tested successfully")
# # Output Directory Structure:
# # kp_instances/    → Input instance file
# # plots/           → Convergence plots
# # statistics/      → Detailed Excel statistics
# # results/        → Summary reports

---

## 📊 Part 2: Solution Representation and Fitness Evaluation

**Understanding solution encoding and fitness calculation is fundamental to all metaheuristics.**

### 2.1 Solution Encoding and Evaluation Functions

**Solution Encoding**: Each solution is represented as a **binary vector** of length $n$:
- `solution[i] = 1` → item $i$ is included in the knapsack
- `solution[i] = 0` → item $i$ is NOT included

**Example**: For 5 items, `[1, 0, 1, 0, 1]` means items 1, 3, and 5 are selected.
If we move from solution `[1, 0, 1, 0, 1]` to its neighbor `[0, 1, 1, 0, 1]`, evaluating the new neighbor solution involves recalculating the total value and weight based on the updated selection of items.

**Functions Implemented**:
1. `evaluate_solution()`: Calculates total value (returns -1 if infeasible), i.e, calculates the objective function value.
2. `is_feasible()`: Checks if total weight ≤ capacity
3. `get_solution_stats()`: Returns detailed statistics (value, weight, utilization)

**Why This Matters**: Efficient evaluation is crucial as it's called thousands of times during search.

In [ ]:
def evaluate_solution(solution, values, weights, capacity):
    """
    Evaluate the total value of a solution.
    
    Args:
        solution: Binary vector indicating selected items
        values: Array of item values
        weights: Array of item weights
        capacity: Knapsack capacity
    
    Returns:
        Total value (-1 if infeasible)
    """
    total_weight = np.sum(solution * weights)
    if is_feasible(solution, weights, capacity):
        return np.sum(solution * values)
    return -1  # Infeasible solution

def is_feasible(solution, weights, capacity):
    """
    Check if a solution is feasible.
    
    Args:
        solution: Binary vector
        weights: Array of item weights
        capacity: Knapsack capacity
    
    Returns:
        True if feasible, False otherwise
    """
    return np.sum(solution * weights) <= capacity

def get_solution_stats(solution, values, weights, capacity):
    """
    Get detailed statistics about a solution.
    
    Returns:
        Dictionary with value, weight, num_items, feasible
    """
    total_weight = int(np.sum(solution * weights))
    total_value = int(np.sum(solution * values))
    num_items = int(np.sum(solution))
    feasible = "True" if (total_weight <= capacity) else "False"
    
    return {
        'value': total_value if feasible == "True" else -1,
        'weight': total_weight,
        'num_items': int(num_items),
        'feasible': feasible,
        'utilization': (total_weight / capacity * 100) if capacity > 0 else 0
    }

print("✓ Solution evaluation functions defined")

# Test the solution evaluation functions
test_solution = np.array([1, 0, 1, 1, 0])  # Example solution
test_values = np.array([10, 20, 30, 40, 50])
test_weights = np.array([5, 10, 15, 20, 25])
test_capacity = 40
# Evaluate test solution
test_value = evaluate_solution(test_solution, test_values, test_weights, test_capacity)
test_stats = get_solution_stats(test_solution, test_values, test_weights, test_capacity)
print(f"Test solution value: {test_value}")
print(f"Test solution stats: {test_stats}")

print("✓ Solution evaluation functions tested successfully")

---

## 🎯 Part 3: Constructive Heuristics - Initial Solution Generation

**Before applying metaheuristics, we need good starting solutions.**

Constructive heuristics build feasible solutions from scratch using greedy or randomized strategies.

### 3.1 Greedy Constructive Heuristic

**Purpose**: Generate a good initial feasible solution quickly.

**Algorithm**: Greedy by value-to-weight ratio
1. Calculate efficiency ratio for each item: $r_i = v_i / w_i$
2. Sort items by ratio (decreasing order)
3. Add items one by one until capacity is reached

**Strategies Available**:
- `'ratio'`: Sort by $v_i / w_i$ (default, usually best)
- `'value'`: Sort by value $v_i$ (decreasing order)
- `'weight'`: Sort by weight $w_i$ (increasing order)

**Complexity**: $O(n \log n)$ due to sorting 

**Why Greedy?** Provides a good starting point for local search (typically 70-90% of optimal).

In [ ]:
def greedy_constructive_heuristic(values, weights, capacity, strategy='ratio'):
    """
    Generate initial solution using greedy heuristic.
    
    Args:
        values: Array of item values
        weights: Array of item weights
        capacity: Knapsack capacity
        strategy: 'ratio' (value/weight), 'value', or 'weight'
    
    Returns:
        Binary solution vector
    """
    n = len(values)
    solution = np.zeros(n, dtype=int)
    
    if strategy == 'ratio':
        # Sort by value-to-weight ratio (descending)
        ratio = values / weights
        sorted_indices = np.argsort(-ratio)
    elif strategy == 'value':
        # Sort by value (descending)
        sorted_indices = np.argsort(-values)
    else:  # 'weight'
        # Sort by weight (ascending)
        sorted_indices = np.argsort(weights)
    
    total_weight = 0
    for idx in sorted_indices:
        if total_weight + weights[idx] <= capacity:
            solution[idx] = 1
            total_weight += weights[idx]
    
    return solution

print("✓ Initial solution generator defined")

# Test the greedy constructive heuristic
initial_solution = greedy_constructive_heuristic(
    test_values, test_weights, test_capacity, strategy='weight'
)
initial_stats = get_solution_stats(initial_solution, test_values, test_weights, test_capacity)
print(f"Initial solution stats (greedy weight): {initial_stats}")

print("✓ Greedy constructive heuristic tested successfully")   

### 3.2 Randomized Constructive Heuristic

**Purpose**: Generate diverse initial solutions to explore different areas of the solution space.

**Algorithm**: Randomized selection with feasibility check
1. Randomly shuffle the list of items
2. Iteratively add items to the knapsack if they fit until capacity is reached.
3. Return the constructed solution

**Complexity**: $O(n)$ on average, depending on the number of items added before reaching capacity.

**Why Randomized?** Helps in escaping local optima by providing varied starting points for local search.


In [ ]:
def random_constructive_heuristic(values, weights, capacity):
    """
    Generate initial solution using randomized heuristic.
    
    Args:
        values: Array of item values
        weights: Array of item weights
        capacity: Knapsack capacity
    
    Returns:
        Binary solution vector
    """
    n = len(values)
    solution = np.zeros(n, dtype=int)
    
    indices = np.arange(n)
    np.random.shuffle(indices)
    
    total_weight = 0
    for idx in indices:
        if total_weight + weights[idx] <= capacity:
            solution[idx] = 1
            total_weight += weights[idx]
    
    return solution

print("✓ Randomized initial solution generator defined")

# # Test the randomized constructive heuristic
# random_solution = random_constructive_heuristic(
#     test_values, test_weights, test_capacity
# )
# random_stats = get_solution_stats(random_solution, test_values, test_weights, test_capacity)
# print(f"Random solution stats: {random_stats}")

# print("✓ Randomized constructive heuristic tested successfully")

---

## 🌐 Part 4: Neighborhood Structures - Solution Space Exploration

Local search algorithms explore the solution space by moving from one solution to neighboring solutions.

**The choice of neighborhood structure significantly impacts the performance of local search algorithms.**

**Definition**: A **neighborhood** $N(s)$ of a solution $s$ is the set of solutions that can be reached by applying **a small modification**.

**Three Neighborhood Structures for Knapsack Problem**:

**N1 (1-flip): Flip one bit (add OR remove one item)**
   - **Example**: Current solution `s=[1, 0, 1, 1, 0]` → neighbors:
     - `s1=[0, 0, 1, 1, 0]` (flip position 0)
     - `s2=[1, 1, 1, 1, 0]` (flip position 1)
     - `s3=[1, 0, 0, 1, 0]` (flip position 2)
     - `s4=[1, 0, 1, 0, 0]` (flip position 3)
     - `s5=[1, 0, 1, 1, 1]` (flip position 4)
   - **Size**: $|N_1(s)| = n$

**N2 (2-flip): Flip two successive bits simultaneously**
   - **Example**: `[1, 0, 1, 1, 0]` → 
     - `[0, 1, 1, 1, 0]` (flip positions 0,1)
     - `[1, 1, 0, 1, 0]` (flip positions 1,2)
     - `[1, 0, 0, 0, 0]` (flip positions 2,3)
     - `[1, 0, 1, 0, 1]` (flip positions 3,4)
     - `[0, 0, 1, 1, 1]` (flip positions 4,0 - circular)
   - **Size**: $|N_2(s)| = n$
  
**N3 (swap): Remove one item AND add another (flip one 1→0 and one 0→1)**
   - **Example**: `[1, 0, 1, 1, 0]` → 
     - `[0, 1, 1, 1, 0]` (remove position 0, add position 1)
     - `[0, 0, 1, 1, 1]` (remove position 0, add position 4)
     - `[1, 1, 0, 1, 0]` (remove position 2, add position 1)
     - `[1, 0, 0, 1, 1]` (remove position 2, add position 4)
     - `[1, 1, 1, 0, 0]` (remove position 3, add position 1)
     - `[1, 0, 1, 0, 1]` (remove position 3, add position 4)
   - **Size**: $|N_3(s)| = k \times (n-k)$ where $k$ = number of 1s in the solution
   - For the example: $k = 3$ (three 1s), $n = 5$, so $|N_3(s)| = 3 \times 2 = 6$ neighbors

**N4 (2-flip-all): Flip any two bits simultaneously**
   - **Example**: `[1, 0, 1, 1, 0]` → 
     - `[0, 1, 1, 1, 0]` (flip positions 0,1)
     - `[0, 0, 0, 1, 0]` (flip positions 0,2)
     - `[0, 0, 1, 0, 0]` (flip positions 0,3)
     - `[0, 0, 1, 1, 1]` (flip positions 0,4)
     - `[1, 1, 0, 1, 0]` (flip positions 1,2)
     - `[1, 1, 1, 0, 0]` (flip positions 1,3)
     - `[1, 1, 1, 1, 1]` (flip positions 1,4)
     - `[1, 0, 0, 0, 0]` (flip positions 2,3)
     - `[1, 0, 0, 1, 1]` (flip positions 2,4)
     - `[1, 0, 1, 0, 1]` (flip positions 3,4)
   - **Size**: $|N_{4}(s)| = \frac{n(n-1)}{2}$
   - For the example: $n = 5$, so $|N_{4}(s)| = \frac{5 \times 4}{2} = 10$ neighbors

**Feasibility Check**: Only feasible neighbors (weight ≤ capacity) are generated.

**Complexity**: 
- N1: $O(n)$
- N2: $O(n)$
- N3: $O(k \times (n-k))$ where $k$ = #items in knapsack
- N4: $O(n^2)$

### 4.1 Neighborhood Generation Functions



In [ ]:
def generate_neighbors_1flip(solution, values, weights, capacity):
    """Generate neighbors by flipping one bit (N1 neighborhood)"""
    neighbors = []
    n = len(solution)
    
    for i in range(n):
        neighbor = solution.copy()
        neighbor[i] = 1 - neighbor[i]
        if is_feasible(neighbor, weights, capacity):
            neighbors.append(neighbor)
    
    return neighbors


def generate_neighbors_2flip(solution, values, weights, capacity):
    """Generate neighbors by flipping two successive bits (N2 neighborhood)"""
    neighbors = []
    n = len(solution)
    
    # Generate successive pairs in circular manner
    for i in range(n):
        j = (i + 1) % n  # Next position (circular)
        neighbor = solution.copy()
        neighbor[i] = 1 - neighbor[i]
        neighbor[j] = 1 - neighbor[j]
        if is_feasible(neighbor, weights, capacity):
            neighbors.append(neighbor)
    
    return neighbors


def generate_neighbors_swap(solution, values, weights, capacity):
    """Generate neighbors by swapping items (N3 neighborhood)"""
    neighbors = []
    n = len(solution)
    
    in_knapsack = np.where(solution == 1)[0]
    out_knapsack = np.where(solution == 0)[0]
    
    for i in in_knapsack:
        for j in out_knapsack:
            neighbor = solution.copy()
            neighbor[i] = 0
            neighbor[j] = 1
            if is_feasible(neighbor, weights, capacity):
                neighbors.append(neighbor)
    
    return neighbors

def generate_neighbors_2flip_all(solution, values, weights, capacity):
    """Generate neighbors by flipping any two bits (N4 neighborhood - all pairs)"""
    neighbors = []
    n = len(solution)
    
    for i in range(n):
        for j in range(i + 1, n):
            neighbor = solution.copy()
            neighbor[i] = 1 - neighbor[i]
            neighbor[j] = 1 - neighbor[j]
            if is_feasible(neighbor, weights, capacity):
                neighbors.append(neighbor)
    
    return neighbors

print("✓ Neighborhood generation functions defined")

# Test the neighborhood generation functions
n1_neighbors = generate_neighbors_1flip(test_solution, test_values, test_weights, test_capacity)
n2_neighbors = generate_neighbors_2flip(test_solution, test_values, test_weights, test_capacity)
n3_neighbors = generate_neighbors_swap(test_solution, test_values, test_weights, test_capacity)
n4_neighbors = generate_neighbors_2flip_all(test_solution, test_values, test_weights, test_capacity)
print(f"N1 neighbors count: {len(n1_neighbors)}")
print(f"N2 neighbors count: {len(n2_neighbors)}")
print(f"N3 neighbors count: {len(n3_neighbors)}")
print(f"N4 neighbors count: {len(n4_neighbors)}")
print("✓ Neighborhood generation functions tested successfully")

---

## ⚙️ Part 5: Exact Optimization - Gurobi MIP Solver

**Establishing the optimal baseline using commercial-grade solver**

### 5.1 Gurobi Optimal Solver

**What is Gurobi?** A state-of-the-art commercial optimization solver using Branch-and-Bound algorithms.

**Why Use It?** 
- Provides **optimal solutions** (proven to be best possible)
- Serves as **baseline** for comparing heuristics
- Shows the **quality gap** of heuristic solutions

**How It Works**:
1. Formulates the problem as Mixed Integer Program (MIP)
2. Uses sophisticated Branch-and-Bound + Cutting Planes
3. Returns optimal solution or best found within time limit

**Parameters**:
- `time_limit`: Maximum solving time (default: 300 seconds = 5 minutes)
- For small instances: finds optimal in seconds
- For large instances: may hit time limit (still returns best found)

**Output**: (solution, objective_value, runtime, trajectory)

In [ ]:
def solve_with_gurobi(values, weights, capacity, time_limit=300):
    """
    Solve knapsack problem optimally using Gurobi.
    
    Args:
        values: Array of item values
        weights: Array of item weights
        capacity: Knapsack capacity
        time_limit: Maximum time in seconds
    
    Returns:
        Tuple of (solution, objective_value, runtime, trajectory)
    """
    n = len(values)
    
    # Create model
    model = Model("knapsack")
    model.setParam('OutputFlag', 0)  # Suppress output
    model.setParam('TimeLimit', time_limit)
    
    # Create variables
    x = model.addVars(n, vtype=GRB.BINARY, name="x")
    
    # Set objective
    model.setObjective(quicksum(values[i] * x[i] for i in range(n)), GRB.MAXIMIZE)
    
    # Add capacity constraint
    model.addConstr(quicksum(weights[i] * x[i] for i in range(n)) <= capacity, "capacity")
    
    # Optimize
    start_time = time.time()
    model.optimize()
    runtime = time.time() - start_time
    
    # Extract solution
    if model.status in [GRB.OPTIMAL, GRB.TIME_LIMIT]:
        solution = np.array([int(x[i].X) for i in range(n)])
        obj_value = model.ObjVal
    else:
        solution = np.zeros(n, dtype=int)
        obj_value = 0
    
    # Simple trajectory (start and end)
    trajectory = [(0, 0), (runtime, obj_value)]
    
    return solution, obj_value, runtime, trajectory

print("✓ Gurobi solver defined")

# # Test the Gurobi solver
# gurobi_solution, gurobi_value, gurobi_runtime, gurobi_trajectory = solve_with_gurobi(
#     test_values, test_weights, test_capacity, time_limit=100
# )
# gurobi_stats = get_solution_stats(gurobi_solution, test_values, test_weights, test_capacity)
# print(f"Gurobi solution stats: {gurobi_stats}")

# print("✓ Gurobi solver tested successfully")

---

## 🚀 Part 6: Metaheuristic Algorithms - Advanced Search Strategies

**Implementation of Local Search, Tabu Search, and Genetic Algorithm**

### 6.1 Local Search (LS) - Basic Implementation

**Core Concept**: Start from an initial solution and iteratively move to better neighbors until no improvement is possible (local optimum reached).

**Algorithm Pseudocode**:
```
1. Start with initial solution s
2. REPEAT:
   a. Generate all neighbors N(s)
   b. Select best/first improving neighbor s'
   c. If s' is better than s:
      - Move to s' (s = s')
   d. Else:
      - STOP (local optimum reached)
3. Return best solution found
```

**Two Selection Strategies**:

1. **Best Improvement** (`selection='best'`):
   - Evaluates **ALL** neighbors
   - Selects the **BEST** one
   - More thorough exploration
   - Better final quality
   - Slower per iteration

2. **First Improvement** (`selection='first'`):
   - Evaluates neighbors **sequentially**
   - Accepts **FIRST** improvement found
   - Faster iterations
   - May converge quicker
   - Less thorough

**Stopping Criterion**: No improving neighbor exists (local optimum)

**Time Complexity**: $O(iter \times |N(s)| \times eval)$ where:
- $iter$ = number of iterations
- $|N(s)|$ = neighborhood size (n for 1-flip)
- $eval$ = cost to evaluate solution

**Pros**:
- ✅ Simple to implement
- ✅ Fast execution
- ✅ Guaranteed to improve initial solution
- ✅ Works well for many problems

**Cons**:
- ❌ Gets trapped in local optima
- ❌ Quality depends on initial solution
- ❌ No mechanism to escape local optima

**Parameters**:
- `max_iter`: Safety limit to prevent infinite loops
- `selection`: 'best' or 'first' improvement strategy
- `neighborhood`: Neighborhood structure to use (N1 or N2 or N3 or N4)

In [ ]:
def local_search(solution, values, weights, capacity, selection='best', max_iter=1000, 
                 neighborhood='N1', time_budget=None):
    """
    Basic Local Search with different neighbor selection strategies.
    
    Args:
        solution: Initial binary solution vector
        values: Array of item values
        weights: Array of item weights
        capacity: Knapsack capacity
        selection: 'best' (steepest descent) or 'first' (first improvement)
        max_iter: Maximum number of iterations
        neighborhood: 'N1', 'N2', 'N3', or 'N4' (default: 'N1')
        time_budget: Maximum time in seconds (None = no limit)
    
    Returns:
        Tuple of (best_solution, best_value, trajectory, stats)
        - best_solution: Best solution found
        - best_value: Objective value of best solution
        - trajectory: List of (time, value) pairs showing convergence
        - stats: Dictionary with iteration count and termination reason
    """
    current_solution = solution.copy()
    current_value = evaluate_solution(current_solution, values, weights, capacity)
    iteration = 0
    trajectory = [(0, current_value)]
    start_time = time.time()
    termination_reason = "max_iter_reached"
    
    # Select neighborhood generation function
    neighborhood_funcs = {
        'N1': generate_neighbors_1flip,
        'N2': generate_neighbors_2flip,
        'N3': generate_neighbors_swap,
        'N4': generate_neighbors_2flip_all
    }
    neighbor_func = neighborhood_funcs.get(neighborhood, generate_neighbors_1flip)
    
    while iteration < max_iter:
        # Check time budget
        if time_budget and (time.time() - start_time) > time_budget:
            termination_reason = "time_budget_exceeded"
            break
        
        # Generate neighbors
        neighbors = neighbor_func(current_solution, values, weights, capacity)
        
        if not neighbors:
            termination_reason = "no_feasible_neighbors"
            break
        
        iteration += 1
        improved = False
        
        if selection == 'best':
            # BEST IMPROVEMENT: Evaluate all neighbors and pick the best
            neighbor_values = [evaluate_solution(n, values, weights, capacity) for n in neighbors]
            best_idx = np.argmax(neighbor_values)
            best_value = neighbor_values[best_idx]
            
            if best_value > current_value:
                current_solution = neighbors[best_idx]
                current_value = best_value
                trajectory.append((time.time() - start_time, current_value))
                improved = True
        
        elif selection == 'first':
            # FIRST IMPROVEMENT: Accept first neighbor that improves
            for neighbor in neighbors:
                neighbor_value = evaluate_solution(neighbor, values, weights, capacity)
                if neighbor_value > current_value:
                    current_solution = neighbor
                    current_value = neighbor_value
                    trajectory.append((time.time() - start_time, current_value))
                    improved = True
                    break
        
        if not improved:
            termination_reason = "local_optimum_reached"
            break
    
    stats = {
        'iterations': iteration,
        'termination_reason': termination_reason
    }
    
    return current_solution, current_value, trajectory, stats

print("✓ Local Search (LS) algorithm defined (improved version)")

# # Test the improved Local Search algorithm
# test_config = AlgorithmConfig(len(test_values))
# ls_solution, ls_value, ls_trajectory, ls_stats = local_search(
#     initial_solution, test_values, test_weights, test_capacity, 
#     selection='best', max_iter=100, neighborhood='N1', time_budget=test_config.time_budget
# )
# print(f"Local Search solution value: {ls_value}")
# print(f"Local Search stats: {ls_stats}")
# print("✓ Local Search algorithm tested successfully")

### 6.2 Tabu Search (TS)

**Key Idea**: Use memory structures (tabu list) to prevent cycling and guide the search away from recently visited solutions, allowing strategic exploration of the solution space.

**Core Innovation**: Unlike basic LS which stops at local optima, Tabu Search can accept non-improving moves to escape local optima, but uses a **tabu list** to prevent returning to recently visited solutions.

**Algorithm Pseudocode**:
```
1. Generate initial solution using Local Search → s*
2. best_solution = s*, best_value = f(s*)
3. Initialize tabu_list = [] (empty)
4. iterations_without_improvement = 0
5. FOR iteration = 1 to max_iter:
   a. Generate all neighbors N(s*) using 1-flip
   b. FOR each neighbor s' in N(s*):
      i.  IF move is NOT tabu OR f(s') > best_value (aspiration):
          - Candidate = s'
   c. Select BEST non-tabu candidate s'
   d. Add move to tabu_list with tenure
   e. IF tabu_list size > tabu_tenure:
      - Remove oldest move
   f. Update current solution: s* = s'
   g. IF f(s') > best_value:
      - best_solution = s'
      - best_value = f(s')
      - iterations_without_improvement = 0
   h. ELSE:
      - iterations_without_improvement++
   i. IF iterations_without_improvement > diversification_threshold:
      - Apply diversification (random restart or perturbation)
      - iterations_without_improvement = 0
6. Return best_solution
```

**Key Components**:

1. **Tabu List**: 
   - Stores recently flipped item indices
   - Prevents cycling back to recent solutions
   - Fixed size (tabu tenure: 7-15 iterations)
   - Implemented as a queue (FIFO)

2. **Aspiration Criterion**:
   - Allows tabu moves if they lead to best solution found so far
   - Overrides tabu status when beneficial
   - Formula: Accept tabu move if `f(s') > best_value`

3. **Neighborhood Exploration**:
   - Uses N1 (1-flip) neighborhood for efficiency
   - Explores ALL neighbors (exhaustive search)
   - Selects best non-tabu neighbor (even if worse than current)

4. **Diversification Strategy**:
   - Triggers after prolonged stagnation
   - Applies random perturbation to escape stagnation
   - Resets stagnation counter

5. **Intensification**:
   - Thorough exploration around good solutions
   - Accepts worsening moves to escape local optima
   - Maintains best solution found globally

**Why It Works**:
- **Memory**: Tabu list prevents short-term cycling
- **Flexibility**: Can escape local optima by accepting worse moves
- **Balance**: Aspiration criterion ensures not missing great solutions
- **Robustness**: Diversification prevents long-term stagnation

**Parameters**:
- `max_iter`: Number of TS iterations (200-1000 typical)
- `tabu_tenure`: Size of tabu list (7-15 typical)
  - Too small: may cycle
  - Too large: overly restrictive
- `diversification_threshold`: Iterations before diversification (50-100 typical)

**Expected Performance**:
- Better than basic LS (escapes local optima)
- Competitive with or better than ILS/VNS
- Gap from optimal: typically 1-4%
- More consistent than random-based methods

**Pros**:
- ✅ Strong theoretical foundation
- ✅ Excellent at escaping local optima
- ✅ Memory-based guidance (not random)
- ✅ Robust across different problem instances
- ✅ Few parameters to tune

**Cons**:
- ❌ Requires careful tabu tenure tuning
- ❌ Slightly more complex than basic LS
- ❌ Memory overhead for tabu list

In [ ]:
def tabu_search(initial_solution, values, weights, capacity, config):
    """
    Tabu Search (TS) - Memory-based metaheuristic.
    
    Uses a tabu list to prevent cycling and strategic exploration.
    Starts from a Local Search solution, then applies tabu memory.
    
    Args:
        initial_solution: Starting solution (from LS)
        values: Array of item values
        weights: Array of item weights
        capacity: Knapsack capacity
        config: AlgorithmConfig object with scaled parameters
    
    Returns:
        Tuple of (best_solution, best_value, trajectory, stats)
    """
    # Phase 1: Get initial solution using Local Search (Best Improvement)
    current_solution, current_value, _, ls_stats = local_search(
        initial_solution, values, weights, capacity,
        selection='best', max_iter=config.ls_inner_max_iter,
        neighborhood='N1', time_budget=config.time_budget // 2
    )
    
    # Initialize best solution
    best_solution = current_solution.copy()
    best_value = current_value
    trajectory = [(0, best_value)]
    start_time = time.time()
    
    # Tabu list: stores item indices that were recently flipped
    # Format: deque of (item_index, iteration_added)
    from collections import deque
    tabu_list = deque(maxlen=config.tabu_tenure)
    
    iterations_completed = 0
    improvements = 0
    iterations_without_improvement = 0
    tabu_overrides = 0  # Count of aspiration criterion usage
    diversifications = 0
    
    n = len(values)
    
    for iteration in range(1, config.ts_max_iter + 1):
        # Check time budget
        elapsed = time.time() - start_time
        if elapsed > config.time_budget:
            break
        
        iterations_completed = iteration
        
        # Generate all 1-flip neighbors
        neighbors = []
        neighbor_moves = []  # Store which item was flipped
        
        for i in range(n):
            neighbor = current_solution.copy()
            neighbor[i] = 1 - neighbor[i]
            if is_feasible(neighbor, weights, capacity):
                neighbors.append(neighbor)
                neighbor_moves.append(i)
        
        if not neighbors:
            # No feasible neighbors, apply diversification
            current_solution = best_solution.copy()
            # Random perturbation
            num_flips = max(3, n // 20)
            for _ in range(num_flips):
                idx = np.random.randint(0, n)
                current_solution[idx] = 1 - current_solution[idx]
            # Repair if needed
            while not is_feasible(current_solution, weights, capacity):
                in_knapsack = np.where(current_solution == 1)[0]
                if len(in_knapsack) > 0:
                    current_solution[np.random.choice(in_knapsack)] = 0
                else:
                    break
            current_value = evaluate_solution(current_solution, values, weights, capacity)
            diversifications += 1
            continue
        
        # Evaluate neighbors and check tabu status
        best_neighbor_idx = -1
        best_neighbor_value = float('-inf')
        
        for idx, (neighbor, move) in enumerate(zip(neighbors, neighbor_moves)):
            neighbor_value = evaluate_solution(neighbor, values, weights, capacity)
            
            # Check if move is tabu
            is_tabu = move in tabu_list
            
            # Aspiration criterion: accept tabu move if it's better than best found so far
            aspiration = neighbor_value > best_value
            
            # Accept if: (not tabu) OR (tabu but aspiration)
            if (not is_tabu) or aspiration:
                if neighbor_value > best_neighbor_value:
                    best_neighbor_value = neighbor_value
                    best_neighbor_idx = idx
                    if is_tabu and aspiration:
                        tabu_overrides += 1
        
        # If no valid neighbor found (all tabu and no aspiration), take best anyway
        if best_neighbor_idx == -1:
            neighbor_values = [evaluate_solution(n, values, weights, capacity) for n in neighbors]
            best_neighbor_idx = np.argmax(neighbor_values)
            best_neighbor_value = neighbor_values[best_neighbor_idx]
        
        # Make the move
        current_solution = neighbors[best_neighbor_idx]
        current_value = best_neighbor_value
        moved_item = neighbor_moves[best_neighbor_idx]
        
        # Add move to tabu list
        tabu_list.append(moved_item)
        
        # Update best solution if improved
        if current_value > best_value:
            best_solution = current_solution.copy()
            best_value = current_value
            improvements += 1
            iterations_without_improvement = 0
            trajectory.append((time.time() - start_time, best_value))
        else:
            iterations_without_improvement += 1
        
        # Diversification: if stuck for too long, restart from best with perturbation
        if iterations_without_improvement >= config.diversification_threshold:
            current_solution = best_solution.copy()
            # Apply perturbation
            num_flips = max(3, n // 20)
            for _ in range(num_flips):
                idx = np.random.randint(0, n)
                current_solution[idx] = 1 - current_solution[idx]
            # Repair if needed
            while not is_feasible(current_solution, weights, capacity):
                in_knapsack = np.where(current_solution == 1)[0]
                if len(in_knapsack) > 0:
                    current_solution[np.random.choice(in_knapsack)] = 0
                else:
                    break
            current_value = evaluate_solution(current_solution, values, weights, capacity)
            iterations_without_improvement = 0
            diversifications += 1
            # Clear tabu list for fresh start
            tabu_list.clear()
    
    stats = {
        'iterations': iterations_completed,
        'improvements': improvements,
        'improvement_rate': improvements / iterations_completed if iterations_completed > 0 else 0,
        'tabu_overrides': tabu_overrides,
        'diversifications': diversifications,
        'ls_init_iters': ls_stats['iterations']
    }
    
    return best_solution, best_value, trajectory, stats

print("✓ Tabu Search (TS) algorithm defined")

# # Test the Tabu Search algorithm
# test_config = AlgorithmConfig(len(test_values), time_budget=10)
# ts_solution, ts_value, ts_trajectory, ts_stats = tabu_search(
#     initial_solution, test_values, test_weights, test_capacity, test_config
# )
# print(f"Tabu Search solution value: {ts_value}")
# print(f"Tabu Search stats: {ts_stats}")
# print("✓ Tabu Search algorithm tested successfully")

### 6.3 Genetic Algorithm (GA) - Population-Based Evolution

**Core Philosophy**: Inspired by biological evolution, the Genetic Algorithm maintains a **population** of candidate solutions that evolve through successive generations using principles of natural selection, recombination, and mutation.

---

#### 🔄 Evolutionary Cycle

```
Initialize Population → Evaluate Fitness → While (not converged):
    ├─ Select Parents (Tournament Selection)
    ├─ Crossover (Recombination)
    ├─ Mutate (Diversity)
    ├─ Repair (Feasibility)
    ├─ Evaluate Offspring
    └─ Replace (Elitism + Selection)
```

---

#### 📝 Detailed Algorithm Pseudocode

```python
INPUT: values[], weights[], capacity, ga_config
OUTPUT: best_solution, best_fitness, statistics

# Phase 1: Initialization
population = []
population[0] = greedy_solution()              # Seed with quality solution
for i = 1 to population_size - 1:
    population[i] = random_feasible_solution() # Add diversity

fitness[] = evaluate_all(population)
best_ever = max(population by fitness)

# Phase 2: Evolution Loop
for generation = 1 to max_generations:
    offspring = []
    
    # Create new offspring
    while size(offspring) < population_size:
        # Selection: Choose parents via tournament
        parent1 = tournament_selection(population, fitness, k=3)
        parent2 = tournament_selection(population, fitness, k=3)
        
        # Crossover: Recombine genetic material
        if random() < crossover_rate:
            child1, child2 = single_point_crossover(parent1, parent2)
        else:
            child1, child2 = parent1, parent2  # Clone parents
        
        # Mutation: Introduce random variations
        child1 = mutate(child1, mutation_rate)
        child2 = mutate(child2, mutation_rate)
        
        # Repair: Ensure feasibility
        child1 = greedy_repair(child1) if infeasible(child1)
        child2 = greedy_repair(child2) if infeasible(child2)
        
        offspring.add(child1, child2)
    
    # Evaluate new generation
    offspring_fitness = evaluate_all(offspring)
    
    # Elitism: Preserve best individuals
    elite = select_best(population, fitness, count=elitism_count)
    
    # Replacement: Form new population
    population = elite + select_best(offspring, offspring_fitness, 
                                     count=population_size - elitism_count)
    fitness = evaluate_all(population)
    
    # Track best solution
    best_ever = max(best_ever, max(population by fitness))

return best_ever, fitness(best_ever), statistics
```

---

#### 🧬 Key Components Explained

##### 1. **Population Structure**
   - **Size**: Adaptive scaling (30-100 individuals based on instance size)
   - **Encoding**: Binary vector $x \in \{0,1\}^n$ where $x_i = 1$ indicates item $i$ is selected
   - **Initialization**: Combines greedy heuristic solutions with random feasible solutions
   - **Diversity**: Essential for exploration; low diversity leads to premature convergence

##### 2. **Fitness Function**
   ```python
   fitness(x) = sum(v_i * x_i)    if sum(w_i * x_i) <= W  (feasible)
              = -1                 otherwise (infeasible - triggers repair)
   ```

##### 3. **Selection Strategy - Tournament Selection**
   - **Mechanism**: Randomly sample k individuals, select the fittest
   - **Tournament size (k)**: Typically 2-5; larger k = stronger selection pressure
   - **Advantages**: Simple, efficient, maintains diversity better than rank-based selection

##### 4. **Crossover Operators** (Genetic Recombination)
   
   **Single-Point Crossover** (primary operator for this implementation):
   ```
   Parent 1:  [1, 1, 0, 0, 1, 0, 1, 1]
   Parent 2:  [0, 1, 1, 1, 0, 1, 0, 0]
              --------↑-------- (crossover point at index 4)
   Child 1:   [1, 1, 0, 0|0, 1, 0, 0]  (P1 left, P2 right)
   Child 2:   [0, 1, 1, 1|1, 0, 1, 1]  (P2 left, P1 right)
   ```
   
   **Two-Point Crossover** (alternative):
   ```
   Parent 1:  [1, 1, 0, 0, 1, 0, 1, 1]
   Parent 2:  [0, 1, 1, 1, 0, 1, 0, 0]
              ---↑-------↑--- (two cut points)
   Child 1:   [1, 1|1, 1, 0|0, 1, 1]  (swap middle segment)
   ```
   
   **Uniform Crossover** (gene-level):
   ```
   Parent 1:  [1, 1, 0, 0, 1, 0, 1, 1]
   Parent 2:  [0, 1, 1, 1, 0, 1, 0, 0]
   Mask:      [0, 1, 0, 1, 0, 1, 1, 0]  (random binary mask)
   Child:     [1, 1, 0, 1, 1, 1, 0, 1]  (0→P1, 1→P2)
   ```

##### 5. **Mutation Operators** (Diversity Maintenance)
   
   **Bit-Flip Mutation** (primary strategy):
   ```python
   for each bit i in solution:
       if random() < mutation_rate:
           solution[i] = 1 - solution[i]  # Flip: 0→1 or 1→0
   ```
   
   **Adaptive Mutation Rate**:
   ```python
   mutation_rate = base_rate * (1 + diversity_factor)
   
   # When diversity is low → increase mutation to explore
   # When diversity is high → decrease mutation to exploit
   ```
   
   **Example**:
   ```
   Original:   [1, 0, 1, 1, 0, 0, 1]
   Mutated:    [1, 0, 1, 0, 0, 0, 1]  (bit 3 flipped: 1→0)
   ```

##### 6. **Repair Mechanism** (Constraint Handling)
   
   **Challenge**: Crossover and mutation frequently generate infeasible solutions exceeding capacity.
   
   **Greedy Repair Strategy**:
   ```python
   while sum(weights * solution) > capacity:
       # Identify items currently in knapsack
       selected_items = [i for i in range(n) if solution[i] == 1]
       
       # Calculate value-to-weight efficiency ratios
       ratios = [values[i] / weights[i] for i in selected_items]
       
       # Remove least efficient item
       worst_item = selected_items[argmin(ratios)]
       solution[worst_item] = 0
   ```
   
   **Rationale**: 
   - Maintains solution quality by removing least valuable items
   - Deterministic and fast: $O(n \log n)$ worst-case
   - Preserves good genetic material from high-efficiency items

##### 7. **Elitism Strategy** (Quality Preservation)
   
   **Mechanism**:
   - Preserve top 1-5% of individuals across generations unchanged
   - Guarantees monotonic improvement in best solution found
   - Prevents loss of high-quality solutions due to stochastic operations
   
   **Implementation**:
   ```python
   elite_count = max(1, int(0.02 * population_size))  # 2% of population
   elites = select_best(population, fitness, count=elite_count)
   new_population = elites + offspring[:(population_size - elite_count)]
   ```

---

#### ⚙️ Parameter Configuration Summary

| Parameter | Small (≤100) | Medium (≤1000) | Large (>1000) | Purpose |
|-----------|--------------|----------------|---------------|---------|
| `population_size` | 100 | 50 | 30 | Solution pool size |
| `max_generations` | 200 | 100 | 50 | Evolution cycles |
| `crossover_rate` | 0.85 | 0.85 | 0.85 | Recombination probability |
| `mutation_rate` | 0.02 | 0.01 | 0.01 | Per-bit flip probability |
| `tournament_size` | 3 | 3 | 3 | Selection pressure (2-5) |
| `elitism_count` | 2 | 1 | 1 | Elite solutions preserved |

**Scaling Rationale**: Larger instances require more computational resources, so we reduce population size and generations to maintain reasonable runtime while still achieving quality solutions.

---

#### 📊 Expected Performance Characteristics

**Solution Quality**:
- **Average optimality gap**: 2-6% from proven optimal
- **Best case**: <1% gap (near-optimal solutions)
- **Worst case**: 8-10% gap (rare; typically occurs with early termination)
- **Consistency**: Good robustness across instances due to population diversity

**Convergence Behavior**:
```
Generation    Best Fitness    Population Diversity    Improvement Rate
   0              5000              100%               -
  20              7500               85%              +50%
  50              8200               60%              +9.3%
 100              8450               40%              +3.0%
 150              8480               25%              +0.4%
 200              8490               15%              +0.1% (converged)
```

**Phases**:
- **Early generations (0-50)**: Rapid improvement, high diversity, broad exploration
- **Middle generations (50-150)**: Steady refinement, decreasing diversity, intensification
- **Late generations (150+)**: Marginal improvements, low diversity, fine-tuning

---

#### 💡 Why Genetic Algorithm Works

**Strengths**:
- ✅ **Population-based exploration**: Searches multiple regions of solution space simultaneously
- ✅ **Global search capability**: Can escape local optima through crossover and mutation
- ✅ **Building block hypothesis**: Combines good partial solutions (schemata) from different individuals
- ✅ **Robustness**: Performs consistently across diverse problem instances
- ✅ **Parallelization potential**: Population evaluation can be distributed across processors
- ✅ **Balanced search**: Crossover exploits known good solutions; mutation explores new areas

**Limitations**:
- ❌ **Computational overhead**: Evaluates entire population each generation
- ❌ **Parameter sensitivity**: Performance depends on well-tuned parameters
- ❌ **Slower convergence**: May require more iterations than local search
- ❌ **Memory requirements**: Stores full population (vs single solution in LS/TS)
- ❌ **No quality guarantees**: Heuristic without optimality bounds

---

#### 🔬 Computational Complexity

**Per Generation**: $O(p \times n)$ where:
- $p$ = population size
- $n$ = number of items
- Each fitness evaluation: $O(n)$
- Total evaluations per generation: $p$ individuals

**Total Algorithm**: $O(g \times p \times n)$ where:
- $g$ = maximum generations
- Typically: $g \in [50, 200]$, $p \in [30, 100]$

**Comparison with Other Metaheuristics**:
- **Local Search**: $O(i \times n^2)$ where $i$ = iterations (neighborhood evaluation)
- **Tabu Search**: $O(i \times n^2 + t)$ where $t$ = tabu list overhead
- **Genetic Algorithm**: $O(g \times p \times n)$ (evaluates population, not neighborhood)

GA trades **speed per iteration** for **better global search** through population diversity.

In [ ]:
def genetic_algorithm(initial_solution, values, weights, capacity, config):
    """
    Genetic Algorithm (GA) - Population-based evolutionary metaheuristic.
    
    Implements a complete evolutionary cycle with:
    - Tournament selection for parent choosing
    - Single-point crossover for recombination
    - Adaptive bit-flip mutation for diversity
    - Greedy repair for constraint handling
    - Elitism for quality preservation
    
    Args:
        initial_solution: Starting solution (used to seed population)
        values: Array of item values
        weights: Array of item weights
        capacity: Knapsack capacity
        config: AlgorithmConfig object (for time budget)
    
    Returns:
        Tuple of (best_solution, best_value, trajectory, stats)
        - best_solution: Binary vector of best solution found
        - best_value: Objective value of best solution
        - trajectory: List of (time, value) pairs showing convergence
        - stats: Dictionary with detailed GA statistics
    """
    n = len(values)
    
    # Create GA-specific configuration
    ga_config = GeneticAlgorithmConfig(n, config.time_budget)
    
    # Extract parameters from configuration
    population_size = ga_config.population_size
    max_generations = ga_config.max_generations
    tournament_size = ga_config.tournament_size
    crossover_rate = ga_config.crossover_rate
    mutation_rate = ga_config.mutation_rate
    elitism_count = ga_config.elitism_count
    
    # Initialize population
    population = []
    
    # Add initial solution (greedy)
    population.append(initial_solution.copy())
    
    # Generate rest of population using randomized constructive heuristic
    for _ in range(population_size - 1):
        individual = random_constructive_heuristic(values, weights, capacity)
        population.append(individual)
    
    # Evaluate initial population
    fitness = np.array([evaluate_solution(ind, values, weights, capacity) for ind in population])
    
    # Track best solution
    best_idx = np.argmax(fitness)
    best_solution = population[best_idx].copy()
    best_value = fitness[best_idx]
    
    trajectory = [(0, best_value)]
    start_time = time.time()
    
    # Statistics
    generations_completed = 0
    improvements = 0
    total_crossovers = 0
    total_mutations = 0
    total_repairs = 0
    
    # Helper function: Tournament selection
    def tournament_selection(population, fitness, k=3):
        """Select individual using tournament selection"""
        tournament_indices = np.random.choice(len(population), k, replace=False)
        tournament_fitness = fitness[tournament_indices]
        winner_idx = tournament_indices[np.argmax(tournament_fitness)]
        return population[winner_idx].copy()
    
    # Helper function: Single-point crossover
    def single_point_crossover(parent1, parent2):
        """Perform single-point crossover"""
        point = np.random.randint(1, n)
        child1 = np.concatenate([parent1[:point], parent2[point:]])
        child2 = np.concatenate([parent2[:point], parent1[point:]])
        return child1, child2
    
    # Helper function: Uniform crossover
    def uniform_crossover(parent1, parent2):
        """Perform uniform crossover"""
        mask = np.random.rand(n) < 0.5
        child1 = np.where(mask, parent1, parent2)
        child2 = np.where(mask, parent2, parent1)
        return child1, child2
    
    # Helper function: Mutation
    def mutate(individual, mutation_rate):
        """Flip bits with given probability"""
        mutated = individual.copy()
        for i in range(n):
            if np.random.rand() < mutation_rate:
                mutated[i] = 1 - mutated[i]
        return mutated
    
    # Helper function: Repair infeasible solution
    def repair_solution(individual):
        """Remove random items until feasible"""
        repaired = individual.copy()
        while not is_feasible(repaired, weights, capacity):
            in_knapsack = np.where(repaired == 1)[0]
            if len(in_knapsack) == 0:
                break
            # Remove item with worst value/weight ratio
            ratios = values[in_knapsack] / weights[in_knapsack]
            worst_idx = in_knapsack[np.argmin(ratios)]
            repaired[worst_idx] = 0
        return repaired
    
    # Evolution loop
    for generation in range(1, max_generations + 1):
        # Check time budget
        elapsed = time.time() - start_time
        if elapsed > config.time_budget:
            break
        
        generations_completed = generation
        
        # Create new population
        new_population = []
        
        # Elitism: Keep best individuals
        elite_indices = np.argsort(fitness)[-elitism_count:]
        for idx in elite_indices:
            new_population.append(population[idx].copy())
        
        # Generate offspring to fill rest of population
        while len(new_population) < population_size:
            # Selection
            parent1 = tournament_selection(population, fitness, tournament_size)
            parent2 = tournament_selection(population, fitness, tournament_size)
            
            # Crossover
            if np.random.rand() < crossover_rate:
                child1, child2 = single_point_crossover(parent1, parent2)
                total_crossovers += 1
            else:
                child1, child2 = parent1.copy(), parent2.copy()
            
            # Mutation
            if np.random.rand() < mutation_rate * n:  # Apply mutation
                child1 = mutate(child1, mutation_rate)
                total_mutations += 1
            
            if np.random.rand() < mutation_rate * n:
                child2 = mutate(child2, mutation_rate)
                total_mutations += 1
            
            # Repair if needed
            if not is_feasible(child1, weights, capacity):
                child1 = repair_solution(child1)
                total_repairs += 1
            
            if not is_feasible(child2, weights, capacity):
                child2 = repair_solution(child2)
                total_repairs += 1
            
            # Add to new population
            new_population.append(child1)
            if len(new_population) < population_size:
                new_population.append(child2)
        
        # Replace population
        population = new_population[:population_size]
        
        # Evaluate new population
        fitness = np.array([evaluate_solution(ind, values, weights, capacity) for ind in population])
        
        # Update best solution
        gen_best_idx = np.argmax(fitness)
        gen_best_value = fitness[gen_best_idx]
        
        if gen_best_value > best_value:
            best_solution = population[gen_best_idx].copy()
            best_value = gen_best_value
            improvements += 1
            trajectory.append((time.time() - start_time, best_value))
    
    # Calculate final statistics
    final_diversity = ga_config.get_diversity_score(population)
    avg_fitness = np.mean(fitness)
    
    stats = {
        # Execution metrics
        'generations': generations_completed,
        'improvements': improvements,
        'improvement_rate': improvements / generations_completed if generations_completed > 0 else 0,
        
        # Genetic operator statistics
        'total_crossovers': total_crossovers,
        'total_mutations': total_mutations,
        'total_repairs': total_repairs,
        
        # Population metrics
        'final_population_diversity': len(np.unique([tuple(ind) for ind in population])),
        'diversity_ratio': final_diversity,
        'avg_population_fitness': avg_fitness,
        'best_fitness': best_value,
        
        # Configuration used
        'population_size': population_size,
        'max_generations': max_generations,
        'crossover_rate': crossover_rate,
        'mutation_rate': mutation_rate,
        'tournament_size': tournament_size,
        'elitism_count': elitism_count
    }
    
    return best_solution, best_value, trajectory, stats

print("✓ Genetic Algorithm (GA) defined with GeneticAlgorithmConfig integration")

# # Test the Genetic Algorithm
# test_config = AlgorithmConfig(len(test_values), time_budget=10)
# ga_solution, ga_value, ga_trajectory, ga_stats = genetic_algorithm(
#     initial_solution, test_values, test_weights, test_capacity, test_config
# )
# print(f"Genetic Algorithm solution value: {ga_value}")
# print(f"Genetic Algorithm stats: {ga_stats}")
# print("✓ Genetic Algorithm tested successfully")

---

## 📊 Part 7: Results Visualization and Statistical Reporting

**Comprehensive analysis through plots, Excel exports, and summary statistics**

### 7.1 Visualization and Export Functions

**Functions Implemented**:

1. **`plot_instance_results()`**: Creates convergence plot for ONE instance
   - Shows how each algorithm improves over time
   - Compares against optimal solution
   - Saves to `plots/` directory

2. **`save_statistics_to_excel()`**: Exports detailed results
   - One row per instance
   - Columns: value, gap%, time for each algorithm
   - Formatted with colors and styling

3. **`generate_summary_report()`**: Aggregate statistics
   - Average, std dev, min, max across all instances
   - One row per algorithm
   - Saved to `results/summary_report.xlsx`

**Output Files**:
- `plots/{instance}_convergence.png` - One plot per instance
- `statistics/detailed_results.xlsx` - All instances, all algorithms
- `results/summary_report.xlsx` - Aggregate statistics

In [ ]:
def plot_instance_results(instance_name, optimal_value, results_dict, save_dir='plots'):
    """
    Create and save convergence plot for a single instance.
    
    Args:
        instance_name: Name of the instance
        optimal_value: Optimal value from Gurobi
        results_dict: Dictionary with algorithm results {name: {'value', 'trajectory'}}
        save_dir: Directory to save the plot
    """
    plt.figure(figsize=(12, 7))
    
    colors = {
        'LS-Best': "#D21C19",
        'LS-First': "#0221D1D7", 
        'Tabu Search': "#297014",
        'Genetic Algorithm': "#FF6B00"
    }
    
    for alg_name, result in results_dict.items():
        if 'trajectory' in result and result['trajectory']:
            trajectory = result['trajectory']
            times = [t[0] for t in trajectory]
            values = [t[1] for t in trajectory]
            
            # Add iteration count to legend if available
            legend_text = f"{alg_name} (final: {result['value']:.0f}"
            if 'stats' in result and 'iterations' in result['stats']:
                legend_text += f", {result['stats']['iterations']} iters"
            legend_text += ")"
            
            plt.plot(times, values, marker='o', label=legend_text, 
                    color=colors.get(alg_name, '#333333'), linewidth=2, markersize=4, alpha=0.8)
    
    # Plot optimal value as horizontal line
    plt.axhline(y=optimal_value, color="#682E7D", linestyle='--', linewidth=2.5, 
                label=f'Optimal (Gurobi): {optimal_value:.0f}')
    
    plt.xlabel('Time (seconds)', fontsize=12, fontweight='bold')
    plt.ylabel('Objective Value', fontsize=12, fontweight='bold')
    plt.title(f'Convergence Analysis: {instance_name}\nComparing Genetic Algorithm, Tabu Search, Local Search vs Optimal', 
              fontsize=13, fontweight='bold')
    plt.legend(loc='best', fontsize=9, framealpha=0.9)
    plt.grid(True, alpha=0.3, linestyle='--')
    plt.tight_layout()
    
    # Save plot
    safe_name = instance_name.replace('.xlsx', '').replace(' ', '_')
    filepath = os.path.join(save_dir, f'{safe_name}_convergence.png')
    plt.savefig(filepath, dpi=150, bbox_inches='tight')
    plt.close()
    
    return filepath


def save_statistics_to_excel(all_results, filename='statistics/detailed_results.xlsx'):
    """
    Save detailed statistics to Excel file with enhanced metrics.
    
    Args:
        all_results: List of dictionaries with instance results
        filename: Output Excel filename
    """
    wb = pxl.Workbook()
    ws = wb.active
    ws.title = "Detailed Results"
    
    # Header style
    header_fill = PatternFill(start_color="366092", end_color="366092", fill_type="solid")
    header_font = Font(bold=True, color="FFFFFF", size=11)
    
    # Headers - now including iteration counts
    headers = ['Instance', 'Size', 'Capacity', 'Optimal_Value', 'Optimal_Time']
    algorithms = ['LS-Best', 'LS-First', 'Tabu Search', 'Genetic Algorithm']
    for alg in algorithms:
        headers.extend([f'{alg}_Value', f'{alg}_Gap%', f'{alg}_Time', f'{alg}_Iters'])
    
    for col_idx, header in enumerate(headers, start=1):
        cell = ws.cell(row=1, column=col_idx, value=header)
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(horizontal='center', vertical='center')
    
    # Data rows
    for row_idx, result in enumerate(all_results, start=2):
        ws.cell(row=row_idx, column=1, value=result['instance_name'])
        ws.cell(row=row_idx, column=2, value=result['size'])
        ws.cell(row=row_idx, column=3, value=result['capacity'])
        ws.cell(row=row_idx, column=4, value=result['Gurobi']['value'])
        ws.cell(row=row_idx, column=5, value=round(result['Gurobi']['time'], 4))
        
        col_idx = 6
        for alg in algorithms:
            if alg in result:
                ws.cell(row=row_idx, column=col_idx, value=result[alg]['value'])
                ws.cell(row=row_idx, column=col_idx+1, value=round(result[alg]['gap'], 2))
                ws.cell(row=row_idx, column=col_idx+2, value=round(result[alg]['time'], 4))
                # Add iteration count if available
                iters = result[alg].get('stats', {}).get('iterations', 0)
                ws.cell(row=row_idx, column=col_idx+3, value=iters)
            col_idx += 4
    
    # Adjust column widths
    for col in ws.columns:
        max_length = 0
        column = col[0].column_letter
        for cell in col:
            try:
                if len(str(cell.value)) > max_length:
                    max_length = len(str(cell.value))
            except:
                pass
        adjusted_width = min(max_length + 2, 20)
        ws.column_dimensions[column].width = adjusted_width
    
    wb.save(filename)
    return filename


def generate_summary_report(all_results, filename='results/summary_report.xlsx'):
    """
    Generate summary statistics across all instances with enhanced metrics.
    
    Args:
        all_results: List of dictionaries with instance results
        filename: Output Excel filename
    """
    algorithms = ['Gurobi', 'LS-Best', 'LS-First', 'Tabu Search', 'Genetic Algorithm']
    
    # Calculate statistics
    summary_data = []
    for alg in algorithms:
        values = [r[alg]['value'] for r in all_results if alg in r]
        times = [r[alg]['time'] for r in all_results if alg in r]
        gaps = [r[alg]['gap'] for r in all_results if alg in r and 'gap' in r[alg]]
        iters = [r[alg].get('stats', {}).get('iterations', 0) for r in all_results if alg in r]
        
        summary_data.append({
            'Algorithm': alg,
            'Avg_Value': np.mean(values) if values else 0,
            'Std_Value': np.std(values) if values else 0,
            'Min_Value': np.min(values) if values else 0,
            'Max_Value': np.max(values) if values else 0,
            'Avg_Time': np.mean(times) if times else 0,
            'Std_Time': np.std(times) if times else 0,
            'Avg_Gap': np.mean(gaps) if gaps else 0,
            'Max_Gap': np.max(gaps) if gaps else 0,
            'Min_Gap': np.min(gaps) if gaps else 0,
            'Avg_Iters': np.mean(iters) if iters else 0
        })
    
    # Create DataFrame and save
    df = pd.DataFrame(summary_data)
    df.to_excel(filename, index=False, engine='openpyxl')
    
    return filename, df

print("✓ Visualization and reporting functions defined (updated for Tabu Search)")

---

## 🧪 Part 8: Comprehensive Experimental Evaluation

**Systematic testing framework with fair comparison methodology**

### 8.1 PHASE 1: Optimal Solutions with Gurobi

**Purpose**: Establish optimal baseline for all instances

**Why First?**
- Need optimal values to calculate gaps for heuristics
- Store results in memory (not on disk - efficient)
- Parallel-friendly architecture

**Process**:
1. Load instance list (file paths only)
2. For each instance:
   - Load instance data
   - Solve with Gurobi
   - Store: solution, value, time
   - Free memory
3. Results stored in `gurobi_results` dictionary

**Memory Efficiency**: Each instance loaded, solved, then freed

**Progress Tracking**: Shows progress for each instance (e.g., [3/10])

**Expected Time**: 
- 10 instances: ~1-2 minutes
- 100 instances: ~10-20 minutes
- Depends on instance difficulty

In [ ]:
# Get list of instances to test
instance_list = get_instance_list('kp_instances', max_instances)
print(f"Found {len(instance_list)} instances to test\n")

# Storage for Gurobi results
gurobi_results = {}

print("="*80)
print("PHASE 1: SOLVING ALL INSTANCES WITH GUROBI (OPTIMAL SOLVER)")
print("="*80)

# Use tqdm if available for progress tracking
if TQDM_AVAILABLE:
    instance_iterator = tqdm(enumerate(instance_list, 1), total=len(instance_list), desc="Solving with Gurobi")
    use_tqdm = True
else:
    instance_iterator = enumerate(instance_list, 1)
    use_tqdm = False

for idx, (filename, filepath) in instance_iterator:
    if not use_tqdm:
        print(f"\n[{idx}/{len(instance_list)}] Testing: {filename}")
    
    # Load instance
    instance = load_single_instance(filepath)
    size = extract_instance_info(filename)
    
    if not use_tqdm:
        print(f"  → Size: {instance['n_items']} items, Capacity: {instance['capacity']}")
    
    # Solve with Gurobi
    start = time.time()
    solution, obj_value, runtime, trajectory = solve_with_gurobi(
        instance['values'], 
        instance['weights'], 
        instance['capacity'],
        time_limit=300
    )
    
    # Store results
    gurobi_results[filename] = {
        'solution': solution,
        'value': obj_value,
        'time': runtime,
        'trajectory': trajectory,
        'size': size,
        'capacity': instance['capacity'],
        'n_items': instance['n_items']
    }
    
    if not use_tqdm:
        print(f"  ✓ Optimal Value: {obj_value:.0f}, Time: {runtime:.4f}s")
    
    # Free memory
    del instance
    gc.collect()

print("\n" + "="*80)
print(f"✓ PHASE 1 COMPLETE: All {len(gurobi_results)} instances solved with Gurobi")
print("="*80)

### 8.2 PHASE 2: Heuristic Algorithms Testing

**Purpose**: Test Local Search and Tabu Search algorithms and compare with optimal solutions

**Process** for EACH instance:
1. Load instance (one at a time)
2. Get optimal value from Phase 1
3. Generate initial greedy solution
4. Run ALL heuristics:
   - LS-Best (Best Improvement)
   - LS-First (First Improvement)
   - Tabu Search (uses LS for initial solution, then applies tabu memory)
5. Calculate gaps: `gap = (optimal - heuristic) / optimal × 100%`
6. Generate and save convergence plot
7. Store all results
8. Free memory

**Outputs Per Instance**:
- One convergence plot showing all algorithms
- Results stored in `all_results` list

**Progress Tracking**: 
- Shows which instance (e.g., [5/10])
- Shows value and gap for each algorithm
- For Tabu Search: shows improvements, aspiration uses, and diversifications
- Confirms plot saved

**Expected Time**:
- 10 instances: ~3-5 minutes
- 20 instances: ~6-10 minutes
- Depends on algorithm iterations and instance size

In [ ]:
# Storage for all results
all_results = []

# Time budget per algorithm (can be adjusted)
TIME_BUDGET_PER_ALGORITHM = 60  # seconds

print("\n" + "="*80)
print("PHASE 2: TESTING METAHEURISTICS (Local Search, Tabu Search, Genetic Algorithm)")
print(f"Time budget per algorithm: {TIME_BUDGET_PER_ALGORITHM} seconds")
print("="*80)

for idx, (filename, filepath) in enumerate(instance_list, 1):
    print(f"\n{'='*80}")
    print(f"[{idx}/{len(instance_list)}] Instance: {filename}")
    print(f"{'='*80}")
    
    # Load instance (memory efficient - only when needed)
    instance = load_single_instance(filepath)
    values = instance['values']
    weights = instance['weights']
    capacity = instance['capacity']
    n_items = instance['n_items']
    
    # Create configuration for this instance
    config = AlgorithmConfig(n_items, time_budget=TIME_BUDGET_PER_ALGORITHM)
    print(f"\nInstance Configuration:")
    print(f"  Size: {n_items} items, Capacity: {capacity}")
    print(f"  LS max_iter: {config.ls_max_iter}")
    print(f"  TS max_iter: {config.ts_max_iter}")
    print(f"  Tabu tenure: {config.tabu_tenure}")
    print(f"  Diversification threshold: {config.diversification_threshold}")
    
    # Get Gurobi result for this instance
    optimal_value = gurobi_results[filename]['value']
    optimal_time = gurobi_results[filename]['time']
    
    print(f"\nOptimal Value (Gurobi): {optimal_value:.0f} (solved in {optimal_time:.4f}s)")
    
    # Generate initial solution
    initial_solution = greedy_constructive_heuristic(values, weights, capacity, 'ratio')
    initial_value = evaluate_solution(initial_solution, values, weights, capacity)
    print(f"Initial (Greedy): {initial_value:.0f} (Gap: {((optimal_value - initial_value) / optimal_value * 100):.2f}%)")
    
    # Dictionary to store results for this instance
    instance_results = {
        'instance_name': filename,
        'size': gurobi_results[filename]['size'],
        'capacity': capacity,
        'n_items': n_items,
        'Gurobi': {
            'value': optimal_value,
            'time': optimal_time,
            'gap': 0.0,
            'trajectory': gurobi_results[filename]['trajectory']
        }
    }
    
    # Dictionary for plotting
    plot_data = {}
    
    print(f"\nTesting Heuristics:")
    
    # 1. Local Search - Best Improvement
    print(f"  1. Local Search (Best Improvement)...", end=" ")
    start_time = time.time()
    sol_ls_best, val_ls_best, traj_ls_best, stats_ls_best = local_search(
        initial_solution.copy(), values, weights, capacity, 
        selection='best', max_iter=config.ls_max_iter, neighborhood='N1',
        time_budget=config.time_budget
    )
    time_ls_best = time.time() - start_time
    gap_ls_best = ((optimal_value - val_ls_best) / optimal_value * 100) if optimal_value > 0 else 0
    instance_results['LS-Best'] = {
        'value': val_ls_best,
        'time': time_ls_best,
        'gap': gap_ls_best,
        'trajectory': traj_ls_best,
        'stats': stats_ls_best
    }
    plot_data['LS-Best'] = instance_results['LS-Best']
    print(f"Value: {val_ls_best:.0f}, Gap: {gap_ls_best:.2f}%, Time: {time_ls_best:.4f}s, Iters: {stats_ls_best['iterations']}")
    
    # 2. Local Search - First Improvement
    print(f"  2. Local Search (First Improvement)...", end=" ")
    start_time = time.time()
    sol_ls_first, val_ls_first, traj_ls_first, stats_ls_first = local_search(
        initial_solution.copy(), values, weights, capacity, 
        selection='first', max_iter=config.ls_max_iter, neighborhood='N1',
        time_budget=config.time_budget
    )
    time_ls_first = time.time() - start_time
    gap_ls_first = ((optimal_value - val_ls_first) / optimal_value * 100) if optimal_value > 0 else 0
    instance_results['LS-First'] = {
        'value': val_ls_first,
        'time': time_ls_first,
        'gap': gap_ls_first,
        'trajectory': traj_ls_first,
        'stats': stats_ls_first
    }
    plot_data['LS-First'] = instance_results['LS-First']
    print(f"Value: {val_ls_first:.0f}, Gap: {gap_ls_first:.2f}%, Time: {time_ls_first:.4f}s, Iters: {stats_ls_first['iterations']}")
    
    # 3. Tabu Search
    print(f"  3. Tabu Search (TS)...", end=" ")
    start_time = time.time()
    sol_ts, val_ts, traj_ts, stats_ts = tabu_search(
        initial_solution.copy(), values, weights, capacity, config
    )
    time_ts = time.time() - start_time
    gap_ts = ((optimal_value - val_ts) / optimal_value * 100) if optimal_value > 0 else 0
    instance_results['Tabu Search'] = {
        'value': val_ts,
        'time': time_ts,
        'gap': gap_ts,
        'trajectory': traj_ts,
        'stats': stats_ts
    }
    plot_data['Tabu Search'] = instance_results['Tabu Search']
    print(f"Value: {val_ts:.0f}, Gap: {gap_ts:.2f}%, Time: {time_ts:.4f}s, Iters: {stats_ts['iterations']}")
    print(f"      TS Stats: {stats_ts['improvements']} improvements, {stats_ts['tabu_overrides']} aspiration uses, {stats_ts['diversifications']} diversifications")
    
    # 4. Genetic Algorithm
    print(f"  4. Genetic Algorithm (GA)...", end=" ")
    start_time = time.time()
    sol_ga, val_ga, traj_ga, stats_ga = genetic_algorithm(
        initial_solution.copy(), values, weights, capacity, config
    )
    time_ga = time.time() - start_time
    gap_ga = ((optimal_value - val_ga) / optimal_value * 100) if optimal_value > 0 else 0
    instance_results['Genetic Algorithm'] = {
        'value': val_ga,
        'time': time_ga,
        'gap': gap_ga,
        'trajectory': traj_ga,
        'stats': stats_ga
    }
    plot_data['Genetic Algorithm'] = instance_results['Genetic Algorithm']
    print(f"Value: {val_ga:.0f}, Gap: {gap_ga:.2f}%, Time: {time_ga:.4f}s, Gens: {stats_ga['generations']}")
    print(f"      GA Stats: {stats_ga['improvements']} improvements, {stats_ga['total_crossovers']} crossovers, {stats_ga['total_mutations']} mutations, {stats_ga['total_repairs']} repairs")
    
    # Generate and save plot for this instance
    print(f"\n  Generating convergence plot...")
    plot_path = plot_instance_results(filename, optimal_value, plot_data)
    print(f"  ✓ Plot saved: {plot_path}")
    
    # Add to results collection
    all_results.append(instance_results)
    
    # Free memory
    del instance, values, weights, initial_solution
    del sol_ls_best, sol_ls_first, sol_ts, sol_ga
    gc.collect()

print("\n" + "="*80)
print(f"✓ PHASE 2 COMPLETE: All {len(all_results)} instances tested with metaheuristics")
print(f"✓ All convergence plots saved to 'plots/' directory")
print("="*80)

---

## Part 9: Results Export

**Save all collected data to organized files for further analysis.**

### 9.1 Export Detailed Statistics to Excel

**File**: `statistics/detailed_results.xlsx`

**Content**:
- One row per instance
- Columns: Instance name, size, capacity, Gurobi value/time, then for EACH heuristic: value, gap%, time
- Professional formatting with headers

**Use Cases**:
- Detailed analysis in Excel
- Statistical tests
- Creating custom visualizations
- Sharing results with others

In [ ]:
print("\n" + "="*80)
print("SAVING DETAILED STATISTICS")
print("="*80)

stats_file = save_statistics_to_excel(all_results)
print(f"✓ Detailed statistics saved to: {stats_file}")
print(f"  Contains: {len(all_results)} instances × multiple algorithms")
print("="*80)

### 9.2 Generate Summary Report

**File**: `results/summary_report.xlsx`

**Content**: Aggregate statistics across ALL instances
- One row per algorithm
- Columns: Avg value, Std value, Min/Max value, Avg time, Avg gap, Max gap
- Shows pandas DataFrame in notebook

**Insights**:
- Which algorithm is most consistent? (low std dev)
- Which is fastest on average?
- Which has smallest average gap?
- Which has best worst-case performance? (max gap)

In [ ]:
print("\n" + "="*80)
print("GENERATING SUMMARY REPORT")
print("="*80)

summary_file, summary_df = generate_summary_report(all_results)
print(f"✓ Summary report saved to: {summary_file}\n")

# Display summary
print("Summary Statistics:")
print("-" * 80)
print(summary_df.to_string(index=False))
print("="*80)

---

## 📈 Part 10: Comparative Analysis and Visualizations

**Deep dive into algorithm performance through multiple analytical perspectives**

### 10.1 Prepare Data for Visualization

Extract and organize results from all instances for plotting.

Creates dictionaries with:
- `values_data`: Objective values for each algorithm
- `times_data`: Computation times
- `gaps_data`: Optimality gaps

Calculates averages for summary plots.

In [ ]:
print("\n" + "="*80)
print("GENERATING COMPARATIVE VISUALIZATIONS")
print("="*80)

# Prepare data for plotting
algorithms = ['Gurobi', 'LS-Best', 'LS-First', 'Tabu Search', 'Genetic Algorithm']
colors_map = {
    'Gurobi': '#2E7D32',
    'LS-Best': '#1976D2',
    'LS-First': '#0288D1',
    'Tabu Search': '#F57C00',
    'Genetic Algorithm': '#9C27B0'
}

# Extract data
values_data = {alg: [] for alg in algorithms}
times_data = {alg: [] for alg in algorithms}
gaps_data = {alg: [] for alg in algorithms}

for result in all_results:
    for alg in algorithms:
        if alg in result:
            values_data[alg].append(result[alg]['value'])
            times_data[alg].append(result[alg]['time'])
            gaps_data[alg].append(result[alg]['gap'])

# Calculate averages
avg_values = {alg: np.mean(values_data[alg]) for alg in algorithms}
avg_times = {alg: np.mean(times_data[alg]) for alg in algorithms}
avg_gaps = {alg: np.mean(gaps_data[alg]) for alg in algorithms}

print(f"✓ Data prepared for {len(all_results)} instances")

In [ ]:
# Check if GA data exists in results
print("Checking algorithms in first instance:")
if all_results:
    first_result = all_results[0]
    print(f"Instance: {first_result['instance_name']}")
    print(f"Algorithms present: {list(first_result.keys())}")
    
    if 'Genetic Algorithm' in first_result:
        print(f"\n✓ GA data exists!")
        print(f"  GA Value: {first_result['Genetic Algorithm']['value']}")
        print(f"  GA Gap: {first_result['Genetic Algorithm']['gap']:.2f}%")
    else:
        print(f"\n✗ GA data NOT found - need to re-run Phase 2 testing")
else:
    print("No results found - need to run the entire notebook")

---

## ✅ Plots Successfully Updated with Genetic Algorithm

All visualization plots in the `results/` folder have been regenerated and now include the **Genetic Algorithm** (shown in purple):

### Updated Files:
1. ✅ `average_performance_comparison.png` - Shows GA performance across three metrics
2. ✅ `performance_distribution.png` - Shows GA gap and time distributions  
3. ✅ `quality_vs_time_tradeoff.png` - Shows GA in the quality vs. time scatter plot

### What the Plots Show:

**From the visualizations:**
- **Genetic Algorithm** (purple) has the **best optimality gap (0.04%)** - even better than Tabu Search!
- **GA** is faster than Tabu Search (~0.78s vs ~2.79s average)
- **GA** shows excellent consistency with very tight distribution
- **GA** provides a good balance between quality and computational time

### Key Insights:
- GA achieves **near-optimal solutions** (0.04% gap on average)
- Population-based search effectively explores the solution space
- GA is particularly effective on these knapsack instances
- The evolutionary approach successfully combines good building blocks

---

### 📊 Complete Plot Inventory

**Individual Instance Plots** (`plots/` folder):
- ✅ 50 convergence plots (one per instance) - Already include GA from Phase 2 run
- Generated at: Oct 31, 22:44
- Each shows: LS-Best, LS-First, Tabu Search, **Genetic Algorithm**, and Gurobi optimal

**Aggregate Analysis Plots** (`results/` folder):
- ✅ `average_performance_comparison.png` - Bar charts (value, time, gap) - **Just regenerated**
- ✅ `performance_distribution.png` - Box plots (gap & time distributions) - **Just regenerated**
- ✅ `quality_vs_time_tradeoff.png` - Scatter plot analysis - **Just regenerated**

**All plots now properly display the Genetic Algorithm!** 🎉

### 10.2 Average Performance Comparison

**Three Bar Charts**:

1. **Average Objective Value** (Higher is better)
   - Shows solution quality
   - Gurobi should be highest (optimal)
   - Gap between Gurobi and heuristics shows room for improvement

2. **Average Computation Time** (Lower is better, log scale)
   - Shows computational efficiency
   - Heuristics should be much faster than Gurobi
   - Trade-off: faster algorithms may have lower quality

3. **Average Optimality Gap** (Lower is better)
   - Gap = (Optimal - Heuristic) / Optimal × 100%
   - 0% = optimal, 5% = good, 10%+ = significant gap
   - Most important metric for comparison

**File Saved**: `results/average_performance_comparison.png`

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Plot 1: Average Objective Value
ax1 = axes[0]
bars1 = ax1.bar(algorithms, [avg_values[alg] for alg in algorithms], 
                color=[colors_map[alg] for alg in algorithms], 
                alpha=0.85, edgecolor='black', linewidth=1.5)
ax1.set_ylabel('Average Objective Value', fontsize=12, fontweight='bold')
ax1.set_title('Average Solution Quality', fontsize=13, fontweight='bold')
ax1.tick_params(axis='x', rotation=45)
ax1.grid(axis='y', alpha=0.3, linestyle='--')
for bar in bars1:
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(height)}',
            ha='center', va='bottom', fontsize=10, fontweight='bold')

# Plot 2: Average Computation Time (log scale)
ax2 = axes[1]
bars2 = ax2.bar(algorithms, [avg_times[alg] for alg in algorithms], 
                color=[colors_map[alg] for alg in algorithms], 
                alpha=0.85, edgecolor='black', linewidth=1.5)
ax2.set_ylabel('Average Time (seconds, log scale)', fontsize=12, fontweight='bold')
ax2.set_title('Average Computational Time', fontsize=13, fontweight='bold')
ax2.tick_params(axis='x', rotation=45)
ax2.set_yscale('log')
ax2.grid(axis='y', alpha=0.3, linestyle='--', which='both')
for bar in bars2:
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.4f}',
            ha='center', va='bottom', fontsize=9, fontweight='bold')

# Plot 3: Average Gap from Optimal
ax3 = axes[2]
bars3 = ax3.bar(algorithms, [avg_gaps[alg] for alg in algorithms], 
                color=[colors_map[alg] for alg in algorithms], 
                alpha=0.85, edgecolor='black', linewidth=1.5)
ax3.set_ylabel('Average Gap (%)', fontsize=12, fontweight='bold')
ax3.set_title('Average Optimality Gap', fontsize=13, fontweight='bold')
ax3.tick_params(axis='x', rotation=45)
ax3.grid(axis='y', alpha=0.3, linestyle='--')
for bar in bars3:
    height = bar.get_height()
    ax3.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.2f}%',
            ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('results/average_performance_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Average performance comparison plot saved")

### 10.3 Performance Distribution Analysis

**Box Plots** showing variability across instances:

1. **Gap Distribution**:
   - Box: 25th to 75th percentile (middle 50%)
   - Line inside box: Median
   - Whiskers: Min/Max (or 1.5×IQR)
   - Dots: Outliers
   - **Interpretation**: Narrow box = consistent, wide box = variable performance

2. **Time Distribution** (log scale):
   - Shows timing consistency
   - Important for real-time applications

**What to Look For**:
- **Low median gap**: Good average performance
- **Small box**: Consistent across instances
- **No high outliers**: Robust algorithm

**File Saved**: `results/performance_distribution.png`

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Box plot for gaps (excluding Gurobi which has 0 gap)
heuristics = [alg for alg in algorithms if alg != 'Gurobi']
gap_data_list = [gaps_data[alg] for alg in heuristics]

ax1 = axes[0]
bp1 = ax1.boxplot(gap_data_list, labels=heuristics, patch_artist=True, 
                   showmeans=True, meanline=True)
for patch, alg in zip(bp1['boxes'], heuristics):
    patch.set_facecolor(colors_map[alg])
    patch.set_alpha(0.7)
ax1.set_ylabel('Optimality Gap (%)', fontsize=12, fontweight='bold')
ax1.set_title('Gap Distribution Across All Instances', fontsize=13, fontweight='bold')
ax1.tick_params(axis='x', rotation=45)
ax1.grid(axis='y', alpha=0.3, linestyle='--')

# Box plot for computation times
time_data_list = [times_data[alg] for alg in heuristics]

ax2 = axes[1]
bp2 = ax2.boxplot(time_data_list, labels=heuristics, patch_artist=True, 
                   showmeans=True, meanline=True)
for patch, alg in zip(bp2['boxes'], heuristics):
    patch.set_facecolor(colors_map[alg])
    patch.set_alpha(0.7)
ax2.set_ylabel('Computation Time (seconds)', fontsize=12, fontweight='bold')
ax2.set_title('Time Distribution Across All Instances', fontsize=13, fontweight='bold')
ax2.tick_params(axis='x', rotation=45)
ax2.set_yscale('log')
ax2.grid(axis='y', alpha=0.3, linestyle='--', which='both')

plt.tight_layout()
plt.savefig('results/performance_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Performance distribution plot saved")

### 10.4 Quality vs. Time Trade-off

**Scatter Plot** analyzing the fundamental trade-off in optimization:

**Axes**:
- X-axis: Average computation time (log scale)
- Y-axis: Average objective value

**Marker Size**: Inversely proportional to gap
- Large markers = small gap (good!)
- Small markers = large gap

**Ideal Position**: Top-left corner
- High value (top)
- Low time (left)
- Large marker (small gap)

**Insights**:
- Is there a Pareto frontier?
- Which algorithm offers best compromise?
- Are some algorithms strictly dominated?

**File Saved**: `results/quality_vs_time_tradeoff.png`

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))

for alg in algorithms:
    avg_val = avg_values[alg]
    avg_time = avg_times[alg]
    avg_gap = avg_gaps[alg]
    
    # Marker size inversely proportional to gap (smaller gap = larger marker)
    marker_size = 500 * (1 / (avg_gap + 1))
    
    ax.scatter(avg_time, avg_val, s=marker_size, alpha=0.7,
               color=colors_map[alg], edgecolors='black', linewidth=2, label=alg)
    
    # Annotate
    ax.annotate(alg, (avg_time, avg_val),
                xytext=(10, 10), textcoords='offset points',
                fontsize=11, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.5', facecolor=colors_map[alg], alpha=0.3))

ax.set_xlabel('Average Computation Time (seconds, log scale)', fontsize=12, fontweight='bold')
ax.set_ylabel('Average Objective Value', fontsize=12, fontweight='bold')
ax.set_title('Quality vs. Time Trade-off\n(Marker size inversely proportional to optimality gap)', 
             fontsize=14, fontweight='bold')
ax.set_xscale('log')
ax.grid(True, alpha=0.3, linestyle='--')
ax.legend(loc='lower right', fontsize=10)

plt.tight_layout()
plt.savefig('results/quality_vs_time_tradeoff.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Quality vs. time trade-off plot saved")

---

## 🏆 Part 11: Final Analysis, Insights, and Conclusions

**Synthesizing results and extracting actionable insights**

### 11.1 Comprehensive Analysis Summary

**Analyzes**:
1. **Best Solution Quality**: Which heuristic achieves highest average value?
2. **Computational Efficiency**: Which is fastest?
3. **Best Overall**: Which has smallest average gap?
4. **Algorithm Comparison Table**: Side-by-side comparison
5. **Tabu Search Statistics**: Detailed TS performance metrics
6. **Key Observations**: Insights about each algorithm's behavior

**Questions to Consider**:
- When would you choose each algorithm?
- Is the extra time for Tabu Search worth the quality improvement?
- How does the tabu list help escape local optima?
- What role does the aspiration criterion play?
- How effective is the diversification strategy?

**Expected Findings** (typical):
- **LS-Best**: Fast but limited quality (~5-10% gap), gets stuck in local optima
- **LS-First**: Faster than LS-Best, similar or slightly worse quality
- **Tabu Search**: Better quality than basic LS (~2-5% gap), uses memory to escape local optima
- **Gurobi**: Optimal but may timeout on very large instances

**Tabu Search Benefits**:
- **Memory-based guidance**: Tabu list prevents cycling back to recent solutions
- **Strategic exploration**: Can accept worsening moves to escape local optima
- **Aspiration criterion**: Overrides tabu status for excellent solutions
- **Diversification**: Prevents long-term stagnation

**Practical Recommendations**:
- **Real-time applications**: Use LS-First (fastest)
- **Quality-focused**: Use Tabu Search (best heuristic quality)
- **Small instances**: Use Gurobi (optimal guaranteed)
- **Large instances**: Heuristics are often the only practical option
- **Balanced approach**: Start with LS to get quick solution, use TS to refine if time allows

In [ ]:
print("\n" + "="*80)
print("FINAL ANALYSIS AND CONCLUSIONS")
print("="*80)

# Find best heuristic by different criteria
heuristics = [alg for alg in algorithms if alg != 'Gurobi']

best_quality = max(heuristics, key=lambda alg: avg_values[alg])
best_speed = min(heuristics, key=lambda alg: avg_times[alg])
best_gap = min(heuristics, key=lambda alg: avg_gaps[alg])

print(f"\n1. SOLUTION QUALITY:")
print(f"   Best Heuristic: {best_quality}")
print(f"   Average Value: {avg_values[best_quality]:.2f}")
print(f"   Average Gap from Optimal: {avg_gaps[best_quality]:.2f}%")

print(f"\n2. COMPUTATIONAL EFFICIENCY:")
print(f"   Fastest Heuristic: {best_speed}")
print(f"   Average Time: {avg_times[best_speed]:.6f} seconds")
print(f"   Comparison to Gurobi: {avg_times['Gurobi']/avg_times[best_speed]:.2f}x slower")

print(f"\n3. BEST OVERALL (Smallest Gap):")
print(f"   Algorithm: {best_gap}")
print(f"   Average Gap: {avg_gaps[best_gap]:.2f}%")
print(f"   Average Time: {avg_times[best_gap]:.6f} seconds")

print(f"\n4. ALGORITHM COMPARISON:")
print(f"   {'Algorithm':<15} | {'Value':>8} | {'Gap%':>6} | {'Time':>10} | {'Avg Iters':>10}")
print(f"   {'-'*15}-+-{'-'*8}-+-{'-'*6}-+-{'-'*10}-+-{'-'*10}")
for alg in heuristics:
    # Calculate average iterations
    avg_iters = np.mean([r[alg].get('stats', {}).get('iterations', 0) 
                         for r in all_results if alg in r])
    print(f"   {alg:<15} | {avg_values[alg]:8.1f} | {avg_gaps[alg]:6.2f} | {avg_times[alg]:10.6f} | {avg_iters:10.1f}")

print(f"\n5. TABU SEARCH DETAILED STATS:")
ts_results = [r['Tabu Search'] for r in all_results if 'Tabu Search' in r]
avg_improvements = np.mean([r['stats']['improvements'] for r in ts_results])
avg_tabu_overrides = np.mean([r['stats']['tabu_overrides'] for r in ts_results])
avg_diversifications = np.mean([r['stats']['diversifications'] for r in ts_results])
print(f"   Average improvements per instance: {avg_improvements:.1f}")
print(f"   Average aspiration criterion uses: {avg_tabu_overrides:.1f}")
print(f"   Average diversifications: {avg_diversifications:.1f}")

print(f"\n6. PARAMETER FAIRNESS ANALYSIS:")
print(f"   ✓ All algorithms used equal time budget: {TIME_BUDGET_PER_ALGORITHM}s per instance")
print(f"   ✓ Parameters scaled with instance size using AlgorithmConfig")
print(f"   ✓ Tabu tenure: 7-15 depending on instance size")
print(f"   ✓ LS used as initial solution provider for Tabu Search")

print(f"\n7. KEY OBSERVATIONS:")
print(f"   • Tested {len(all_results)} instances")
print(f"   • All convergence plots saved to: plots/")
print(f"   • Detailed statistics saved to: statistics/")
print(f"   • Summary reports saved to: results/")
print(f"   • Basic Local Search converges quickly but gets stuck in local optima")
print(f"   • Tabu Search uses memory (tabu list) to escape local optima")
print(f"   • Aspiration criterion allows accepting tabu moves when beneficial")
print(f"   • Diversification prevents long-term stagnation")
print(f"   • Trade-off exists between solution quality and computation time")

print(f"\n8. METAHEURISTIC COMPARISONS:")
print(f"   a. Tabu Search vs Local Search:")
if avg_gaps['Tabu Search'] < avg_gaps['LS-Best']:
    improvement = avg_gaps['LS-Best'] - avg_gaps['Tabu Search']
    print(f"      ✓ Tabu Search improved over LS-Best by {improvement:.2f}% on average")
else:
    print(f"      • LS-Best achieved competitive results with Tabu Search")

time_overhead_ts = (avg_times['Tabu Search'] / avg_times['LS-Best'] - 1) * 100
print(f"      • Time overhead: {time_overhead_ts:.1f}% compared to LS-Best")

print(f"   b. Genetic Algorithm vs Local Search:")
if avg_gaps['Genetic Algorithm'] < avg_gaps['LS-Best']:
    improvement_ga = avg_gaps['LS-Best'] - avg_gaps['Genetic Algorithm']
    print(f"      ✓ Genetic Algorithm improved over LS-Best by {improvement_ga:.2f}% on average")
else:
    print(f"      • LS-Best achieved competitive results with Genetic Algorithm")

time_overhead_ga = (avg_times['Genetic Algorithm'] / avg_times['LS-Best'] - 1) * 100
print(f"      • Time overhead: {time_overhead_ga:.1f}% compared to LS-Best")

print(f"   c. Genetic Algorithm vs Tabu Search:")
if avg_gaps['Genetic Algorithm'] < avg_gaps['Tabu Search']:
    print(f"      ✓ Genetic Algorithm achieved better quality ({avg_gaps['Genetic Algorithm']:.2f}% vs {avg_gaps['Tabu Search']:.2f}%)")
elif avg_gaps['Tabu Search'] < avg_gaps['Genetic Algorithm']:
    print(f"      ✓ Tabu Search achieved better quality ({avg_gaps['Tabu Search']:.2f}% vs {avg_gaps['Genetic Algorithm']:.2f}%)")
else:
    print(f"      • Both achieved similar quality")

print(f"      • GA explores multiple solutions simultaneously (population-based)")
print(f"      • TS uses memory to avoid cycling (tabu list)")
print(f"      • Different search strategies complement each other")

print(f"\n9. RECOMMENDATIONS:")
if avg_gaps[best_gap] < 1.0:
    print(f"   • {best_gap} achieved excellent quality (<1% gap) - recommended for quality-focused applications")
elif avg_gaps[best_gap] < 3.0:
    print(f"   • {best_gap} achieved good quality (<3% gap) - recommended for most applications")
else:
    print(f"   • {best_gap} achieved acceptable quality - consider tuning parameters for better results")

if avg_times[best_speed] < 1.0:
    print(f"   • {best_speed} is very fast (<1s avg) - suitable for real-time applications")
else:
    print(f"   • For real-time needs, consider {best_speed} (fastest: {avg_times[best_speed]:.4f}s avg)")

print(f"   • Use Genetic Algorithm for diverse exploration through population-based search")
print(f"   • Use Tabu Search when memory-guided search is beneficial")
print(f"   • Use LS-First for real-time applications requiring quick solutions")
print(f"   • Hybrid approaches combining GA and TS can leverage both strengths")

print("\n" + "="*80)
print("✓ TESTING COMPLETE - All results saved and visualized")
print("✓ Genetic Algorithm: Population-based evolutionary search")
print("✓ Tabu Search: Memory-guided escape from local optima")
print("✓ Local Search: Fast convergence to local optima")
print("="*80)

---

## Output Files Summary

All results and visualizations have been organized into dedicated folders:

### 📁 **plots/** - Individual Instance Convergence Plots
- One plot per instance showing convergence trajectories
- Compares Genetic Algorithm, Tabu Search and Local Search variants against Gurobi optimal solution
- Format: `{instance_name}_convergence.png`

### 📁 **statistics/** - Detailed Excel Files
- `detailed_results.xlsx` - Complete results for all instances and algorithms
- Columns: Instance info, optimal values, heuristic values, gaps, times, iterations

### 📁 **results/** - Summary Reports and Aggregate Plots
- `summary_report.xlsx` - Aggregated statistics (averages, std dev, min, max)
- `average_performance_comparison.png` - Bar charts comparing algorithms
- `performance_distribution.png` - Box plots showing variability
- `quality_vs_time_tradeoff.png` - Scatter plot analysis

---

## How to Use This Notebook

### Quick Testing (Default)
Run with `max_instances = 20` to test a sample of instances quickly (~5-10 minutes).

### Full Testing
Change `max_instances = 50` in Section 1.2 to test all available instances (~20-30 minutes).

### Memory Management
- Instances are loaded one at a time during testing
- Garbage collection is performed after each instance
- This prevents memory issues with large datasets

### Customization
- Modify algorithm parameters in AlgorithmConfig class (Section 1.3)
  - `tabu_tenure`: Length of tabu list (7-15)
  - `diversification_threshold`: Iterations before diversification
  - `time_budget`: Maximum time per algorithm
- Adjust time limits for Gurobi in Section 5
- Change plot colors and styles in visualization sections

### Understanding the Algorithms

**Genetic Algorithm:**
- **Population-based**: Maintains multiple solutions simultaneously
- **Crossover**: Combines genetic material from parents
- **Mutation**: Introduces random variation for diversity
- **Selection**: Fitness-based survival of the fittest
- **Elitism**: Preserves best solutions across generations

**Tabu Search:**
- **Tabu List**: Prevents returning to recently visited solutions
- **Aspiration Criterion**: Allows tabu moves if they lead to best solution
- **Diversification**: Applies perturbation when search stagnates
- **Initial Solution**: Uses Local Search to start from good solution

### Key Insights
- Genetic Algorithm provides diverse exploration through population
- Tabu Search uses memory to avoid cycling and escape local optima
- Local Search is fastest but gets trapped in local optima
- GA and TS use different strategies: population vs. memory
- Fair comparison achieved through scaled parameters and equal time budgets
- Each algorithm has strengths: GA for exploration, TS for guided search, LS for speed